In [30]:
import os
from typing import Dict, List, Literal, Tuple, Optional
import logging
import math

import category_encoders as ce
from catboost import CatBoostRegressor
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from pycatch22 import catch22_all
from scipy.interpolate import PchipInterpolator

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

import xarray as xr
import zarr
import shutil
from glob import glob

from forecasting_module import TimeGPTForecaster, SARIMAXForecaster
from utils import ForecastUtils

from lstm_network import LSTMModel, LSTMTrainer

import autoencoders as ae
import train_autoencoders as train_ae
from other_encoders.ts2vec_encoder import TS2VecEncoder

from predictions import MLPHead, ProjectionHead

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
def plot_time_deltas(df: pd.DataFrame, time_col: str) -> None:
    """Calculate time diff between consecutive datetime entries in the specified column,
    and plot these time deltas"""
    df[time_col]= pd.to_datetime(df[time_col])
    diffs       = df[time_col].diff().dropna()
    plt.plot(diffs.dt.total_seconds())
    plt.ylabel('Time delta (s)')
    plt.title('Time diff Between Consecutive datetimes')
    plt.show()

datasets = {"GDELT_USA_LAB":      {"path": "Time-IMM/GDELT_USA_LAB",       "time_col": "date_time", "y": ["AvgTone"]},
            # "ClusterTrace_707":   {"path": "Time-IMM/ClusterTrace_707",    "time_col": "date_time", "y": ["cpu", "memory"]},
            "EPA_air_los_angeles":{"path": "Time-IMM/EPA_air_los_angeles", "time_col": "date_time", "y": ["ozone"]},
            "FNSPID_EXPE":        {"path": "Time-IMM/FNSPID_EXPE",         "time_col": "date_time", "y": ["adj close"]},
            # "ILINet":             {"path": "Time-IMM/ILINet",              "time_col": "date_time", "y": ["TOTAL PATIENTS"]},
            "repohealth_facebook":{"path": "Time-IMM/repohealth_facebook", "time_col": "date_time", "y": ["new_issues"]},
            "studentlife_fc7337": {"path": "Time-IMM/studentlife_fc7337",  "time_col": "date_time", "y": ["sleep_duration"]},
            "power_consumption":  {"path": "power_consumption",            "time_col": "Datetime",  "y": ["PowerConsumption_Zone1", "PowerConsumption_Zone2", "PowerConsumption_Zone3"]},
            "appliances_energy":  {"path": "appliances_energy",            "time_col": "date",      "y": ["rv1", "rv2"]},
            
            "india_catchment": {"path": "india_catchments"},
            "argoverse":       {"path": "argoverse_forecasting"},
            "weather_bench2":  {"path": "?"},
            "PTB_ECG":         {"path": "ptb-xl-1.0.3"},
            }

desired_dataset= "india_catchment"

df             = pd.read_csv(f"../public_datasets/2D/{datasets[desired_dataset]['path']}.csv")
time_col_name  = datasets[desired_dataset]["time_col"]

"""Pre-processing each dataset"""
try:
    if desired_dataset == "GDELT_USA_LAB":
        df.drop(["record_id"], axis=1, inplace=True)
    elif desired_dataset == "ClusterTrace_707":
        pass
    elif desired_dataset == "EPA_air_los_angeles":
        df.drop(["record_id"], axis=1, inplace=True)
    elif desired_dataset == "FNSPID_EXPE":
        df.drop(["record_id"], axis=1, inplace=True)
    elif desired_dataset == "ILINet":
        df.drop(["record_id"], axis=1, inplace=True)
        df = df.drop_duplicates(subset=['date_time'], keep='first').reset_index(drop=True) #some rows have same time
    elif desired_dataset == "repohealth_facebook":
        df.drop(["record_id"], axis=1, inplace=True)
    elif desired_dataset == "studentlife_fc7337":
        df.drop(["record_id"], axis=1, inplace=True)
    elif desired_dataset == "power_consumption":
        df.drop(["PowerConsumption_Zone2", "PowerConsumption_Zone3"], axis=1, inplace=True)
    elif desired_dataset == "appliances_energy":
        df.drop(["rv2"], axis=1, inplace=True)
except Exception as e:
    pass

y_cols = [col for col in datasets[desired_dataset]["y"] if col in df.columns]
df = df.fillna(0)

plot_time_deltas(df, time_col_name)


In [ ]:
"""Common pre-processing steps"""

# interpolate irregular timestamps first
time_delta = df[time_col_name].diff().dropna()
if time_delta.nunique() == 1: # timestamps uniform, skip PCHIP
    df_uniform = df.copy()
else:
    freq       = ForecastUtils.infer_dominant_freq(df, time_col_name)
    df_uniform = ForecastUtils.apply_pchip_interpolation(df, time_col_name, freq)

horizon  = ForecastUtils.compute_horizon(len(df_uniform), fixed_points=20, pct=0.01)
n_train  = len(df_uniform) - horizon
df_train = df_uniform.iloc[:n_train].copy()
df_test  = df_uniform.iloc[n_train:].copy()

n_windows     = 5    # windows to create 
horizon       = 10   # prediction horizon for each window
horizon_frac  = 0.01 # frac of ENTIRE series length
min_window_len= 1008 # len of smallest window (1008 is timeGPT default)


In [ ]:
"""TimeGPT"""
NIXTLA_API_KEY = 'nixak-TnuMDCHsSM4hajkuXycqZZrNxwtAIoT9O9H7Q8ZwKl2JuJlazRqIPknwJW1AVHX2yB3yfCAwmAogugqQ'
logging.getLogger("nixtla").setLevel(logging.WARNING)

forecaster = TimeGPTForecaster(df_uniform, time_col=time_col_name, y_cols=y_cols, api_key=NIXTLA_API_KEY)

"""single window evaluation"""
print("==== single window evaluation ====")
forecast_dict, _ = forecaster.forecast_timegpt(df_train, df_test, horizon, use_exogenous_cols=True)
forecast_df_temp = forecaster.forecast_dfs[y_cols[0]]

"""Multi-window evaluation"""
print("==== multi window evaluation ====")
use_exogenous_cols=True
windows_list  = ForecastUtils.make_windows(df_uniform, n_windows=n_windows, horizon_len=horizon, 
                                           horizon_frac=horizon_frac, min_window_len=min_window_len, start_point=0)
all_forecasts, mae_list, sizes = {}, [], []

for i, (train_df, test_df) in enumerate(windows_list):
    print(f"Window {i}: train shape {train_df.shape}, test shape {test_df.shape}")
    forecast_dict, y_true_scaled = forecaster.forecast_timegpt(train_df, test_df, horizon_len=horizon, use_exogenous_cols=use_exogenous_cols)
    all_forecasts[f"window_{i}"] = forecast_dict

    # Compute MAE per window
    for j, target in enumerate(y_cols):
        y_true = y_true_scaled[:, j] # shape = H
        y_pred = forecast_dict[target]
        mae    = mean_absolute_error(y_true, y_pred)
        mae_list.append(mae)
        sizes.append(len(y_true))

# Step 3: compute weighted MAE across windows (horizon = weight)
weighted_mae = sum(MAE * n for MAE, n in zip(mae_list, sizes)) / sum(sizes)
std_mae      = math.sqrt(sum((MAE - weighted_mae) ** 2 * n for MAE, n in zip(mae_list, sizes)) / sum(sizes))
print(f"Final MAE: mean ± std= {{{weighted_mae:.4f}}}{{{std_mae:.4f}}}")

# # Plot:
# client.plot(df, forecast_df, time_col=time_col_name, target_col=y_cols[0], level=[80,90])

# show last 2% of the history + all predictions
# n = int(len(df) * 0.02)
# df_tail = df.tail(n)
# forecast_tail = forecast_df[forecast_df[time_col_name] >= df_tail[time_col_name].iloc[0]]
# client.plot(df_tail,forecast_tail,time_col=time_col_name,target_col=y_cols[0],level=[80, 90])


In [ ]:
"""SARIMAX"""
logging.basicConfig(level=logging.INFO)

forecaster = SARIMAXForecaster(df_uniform, time_col=time_col_name, y_cols=y_cols)

print("==== single window evaluation ====")
forecast_dict, y_true_scaled = forecaster.forecast_sarimax(df_train, df_test, horizon, 
                                                           order=(1, 0, 0), seasonal_order=(0, 0, 0, 0))
forecast_df_temp = forecaster.forecast_dfs[y_cols[0]]

print("==== multi window evaluation ====")
use_exogenous_cols = False
windows_list = ForecastUtils.make_windows(df_uniform, n_windows=n_windows, horizon_len=horizon, 
                                          horizon_frac=horizon_frac, min_window_len=min_window_len, start_point=0)
all_forecasts, mae_list, sizes = {}, [], []

for i, (train_df, test_df) in enumerate(windows_list):
    print(f"Window {i}: train shape {train_df.shape}, test shape {test_df.shape}")
    forecast_dict, y_true_scaled = forecaster.forecast_sarimax(train_df, test_df, horizon, order=(1, 0, 0),
                                                               seasonal_order=(0, 0, 0, 0), use_exogenous_cols=use_exogenous_cols)
    all_forecasts[f"window_{i}"] = forecast_dict

    # Compute MAE per window
    for j, target in enumerate(y_cols):
        y_true = y_true_scaled[:, j]
        y_pred = forecast_dict[target]
        mae    = mean_absolute_error(y_true, y_pred)
        mae_list.append(mae)
        sizes.append(len(y_true))

# ---- Weighted MAE ----
weighted_mae = sum(MAE * n for MAE, n in zip(mae_list, sizes)) / sum(sizes)
std_mae      = math.sqrt(sum((MAE - weighted_mae) ** 2 * n for MAE, n in zip(mae_list, sizes)) / sum(sizes))
print(f"Final MAE: mean ± std= {{{weighted_mae:.4f}}}{{{std_mae:.4f}}}")


In [ ]:
"""run AE"""

# === Example params ===
input_rows        = 1000
input_cols        = 20
layer1_dim        = 64
layer2_dim        = 32
latent_dim        = 8
dropout_prob      = 0.1
layer_dims        = [input_cols, 64, 32, latent_dim]  # for FlexibleAE
pred_dim          = 0
projection_dim    = 0
batch_size        = 32
epochs            = 100
lr                = 1e-3
weight_decay      = 1e-5
scheduler_patience= 5
# === Dummy dataset ===
x_train      = torch.randn(input_rows, input_cols)
train_loader = DataLoader(list(zip(x_train, x_train)), batch_size=batch_size, shuffle=True)

# === Simple AE ===
# simple_ae   = ae.SimpleAutoencoder(input_cols, layer1_dim, layer2_dim, latent_dim, dropout_prob).to(device)
# optimizer = torch.optim.AdamW(simple_ae.parameters(), lr=lr, weight_decay=weight_decay)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=scheduler_patience)
trainer   = train_ae.TrainAutoencoder()
# best_loss = trainer.train_autoencoder(device, simple_ae, epochs, train_loader, optimizer, scheduler)
# print("Best training loss (SimpleAE):", best_loss)

# === FlexibleAE example ===
# flexible_ae = ae.FlexibleAutoencoder(layer_dims, pred_dim, dropout_prob, projection_dim, pred_hidden_dim=32, mode="reconstruct").to(device)
# optimizer = torch.optim.AdamW(flexible_ae.parameters(), lr=lr, weight_decay=weight_decay)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=scheduler_patience)
# best_loss = trainer.train_autoencoder(device, flexible_ae, epochs, train_loader, optimizer, scheduler)
# print(f"Best training loss (FlexibleAE): {best_loss:.3f}")

# === VAE ===
val_loader= DataLoader(list(zip(x_train, x_train)), batch_size=batch_size)  # create validation loader
vae       = ae.VAE(input_cols, hidden_dim=64, latent_dim=latent_dim, dropout_prob=dropout_prob).to(device)
optimizer = torch.optim.AdamW(vae.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=scheduler_patience)
trainer   = train_ae.TrainVAE()
best_loss = trainer.train_vae(device, vae, epochs, train_loader, optimizer, scheduler, val_loader=val_loader)
print(f"Best training loss (VAE): {best_loss:.3f}")

In [ ]:
"""ECG dataset"""
from dataset_loaders import ECGLoader

ECG_data_path = "../public_datasets/3D/ptb-xl-1.0.3"
loader        = ECGLoader(ECG_data_path)
X_ecg, y_ecg, sampling_rate, leads = loader.load_dataset(sampling="lr", target="diagnostic_superclass_multi",
                                                         segment_duration_sec=10.0, max_records=100, continuous_target=True)
print(f"X shape: {X_ecg.shape}, X size (MB): {X_ecg.nbytes / 1024**2:.2f}")   # (patients, time, leads)
print(f"y shape: {y_ecg.shape}, y size (MB): {y_ecg.nbytes / 1024**2:.4f}")  # (patients, conditions)


In [ ]:
"""Argoverse dataset"""

def process_argoverse_parquet(scenario_parquet_path: str):
    """Convert Argoverse Parquet scenario to X (3D) and y (2D)
    Follows these rules:
        1. track_id != focal_track_id AND observed = true → include in X
        2. track_id = focal_track_id AND observed = true → include in X
        3. track_id = focal_track_id AND observed = false → include in y only
        4. track_id != focal_track_id AND observed = false → ignore"""
    df       = pd.read_parquet(scenario_parquet_path)
    focal_id = df['focal_track_id'].iloc[0]

    # Remove irrelevant rows (rule 4), but keep focal agent even if unobserved (rule 3)
    df = df[(df['observed'] == True) | (df['track_id'] == focal_id)]

    agent_past_list   = []
    focal_future_list = []

    for track_id, track_data in df.groupby('track_id'):
        xy_pos     = track_data[['position_x', 'position_y']].values
        headings   = track_data['heading'].values
        velocities = track_data[['velocity_x', 'velocity_y']].values
        is_observed= track_data['observed'].values

        agent_past   = []
        focal_future = []

        for t, obs in enumerate(is_observed):
            if obs:  # observed = True → goes into X
                agent_past.append([xy_pos[t][0],
                                   xy_pos[t][1],
                                   headings[t],
                                   velocities[t][0],
                                   velocities[t][1]])
            elif track_id == focal_id:  # observed = False and focal → goes into y
                focal_future.append(xy_pos[t])  # only x, y
        if agent_past:
            agent_past_list.append(np.array(agent_past))
        if focal_future and track_id == focal_id:
            focal_future_list = np.array(focal_future)

    # pad sequences to longest agent
    max_agent_len   = max(len(p) for p in agent_past_list)
    X_agents_padded = np.array([np.pad(p, ((0, max_agent_len-len(p)), (0,0)), 'constant') for p in agent_past_list])

    return X_agents_padded, np.array(focal_future_list)

# Example usage:
argoverse_data_path = "../public_datasets/3D/argoverse_forecasting"
folder_name    = "00a0ec58-1fb9-4a2b-bfd7-f4e5da7a9eff"
file_name      = "scenario_00a0ec58-1fb9-4a2b-bfd7-f4e5da7a9eff.parquet"
X_argo, y_argo = process_argoverse_parquet(f"{argoverse_data_path}/{folder_name}/{file_name}")
print(X_argo.shape, y_argo.shape)


In [ ]:
"""Weather dataset"""

class WeatherDataset:
    """Loads and preprocesses WeatherBench-style datasets for ML.
    Supports multiple output shapes for X and y"""
    def __init__(self, dataset_folder: str, variables_X: list, variables_y: list, freq="6H"):
        self.dataset_folder = dataset_folder
        self.variables_X    = variables_X
        self.variables_y    = variables_y
        self.freq   = freq
        self.X_xarr = None
        self.y_xarr = None

    @staticmethod
    def open_zarr_variable(folder_path: str, varname: str) -> xr.DataArray:
        """Open a folder containing a single variable as xarray.DataArray"""
        arr = zarr.open(folder_path, mode="r")
        time= pd.date_range("1959-01-01", periods=arr.shape[0], freq="6H")
        lat = np.linspace(-90, 90, arr.shape[1])
        lon = np.linspace(0, 360, arr.shape[2], endpoint=False)
        return xr.DataArray(arr, dims=["time", "lat", "lon"], coords={"time": time, "lat": lat, "lon": lon}, name=varname)

    def load_dataset(self):
        """Load all variables into xarray Datasets for X and y"""
        X_data      = {var: self.open_zarr_variable(f"{self.dataset_folder}/{var}", var) for var in self.variables_X}
        y_data      = {var: self.open_zarr_variable(f"{self.dataset_folder}/{var}", var) for var in self.variables_y}
        self.X_xarr = xr.Dataset(X_data)
        self.y_xarr = xr.Dataset(y_data)
        return self.X_xarr, self.y_xarr

    @staticmethod
    def prepare_X_features(X_ds: xr.Dataset, mode: str = "autoencoder", last_timesteps: int = 1) -> np.ndarray:
        """Convert xarray.Dataset of features to NumPy array
        - "autoencoder" [for autoencoder]  -> keep channels and spatial dims (time x channels x lat x lon)
        - "3d" [for 3D CNN]                -> keep channels, flatten spatial dims (time x channels x lat*lon)
        - "lstm" [for LSTM/Transformer]    -> flatten spatial dims, keep time (time x features)
        - "catboost" [for CatBoost]        -> flatten channels and spatial dims for last timestep only (1 x features)"""
        X_arr = np.stack([X_ds[var].values for var in X_ds.data_vars], axis=1)  # (time, channels, lat, lon)
        if mode == "autoencoder":
            return X_arr
        elif mode == "3d":
            return X_arr.reshape(X_arr.shape[0], X_arr.shape[1], -1)  # (time, channels, lat*lon)
        elif mode == "lstm":
            time_dim, channels, lat, lon = X_arr.shape
            return X_arr.reshape(time_dim, channels * lat * lon)
        elif mode == "catboost":
            X_last = X_arr[-last_timesteps:]
            return X_last.reshape(last_timesteps, -1)
        else:
            raise ValueError("mode must be one of ['autoencoder','3d','lstm','catboost']")

    @staticmethod
    def prepare_y_targets(y_ds: xr.Dataset, mode: str = "3d") -> np.ndarray:
        """Convert xarray.Dataset of targets to NumPy array"""
        y_arr  = np.stack([y_ds[var].values for var in y_ds.data_vars], axis=0)  # (vars, time, lat, lon)
        y_last = y_arr[:, -1, :, :]  # take last timestep
        if mode == "3d":
            return y_last  # (vars, lat, lon)
        elif mode == "flatten":
            return y_last.reshape(1, -1)  # (1, vars*lat*lon)
        elif mode == "collapse":
            return y_last.mean(axis=1)  # (vars, lon)
        else:
            raise ValueError("mode must be one of ['3d','flatten','collapse']")

    @staticmethod
    def reshape_for_ml(X: np.ndarray) -> np.ndarray:
        """Convert X from (time, channels, lat*lon) -> (lat*lon, time, channels)
        for per-grid-cell sequence models like timeVAE."""
        return np.moveaxis(X, [0, 1, 2], [1, 2, 0])

    @staticmethod
    def flatten_y(y: np.ndarray) -> np.ndarray:
        """Flatten y from (vars, lat, lon) -> (lat*lon, vars)
        to align with reshaped X."""
        return y.reshape(y.shape[0], -1).T

dataset_folder = "../public_datasets/3D/weather_bench"
variables_X    = ["2m_temperature", "10m_u_component_of_wind", "10m_v_component_of_wind"]
variables_y    = ["mean_sea_level_pressure", "total_precipitation_6hr"]

weather        = WeatherDataset(dataset_folder, variables_X, variables_y)
X_xarr, y_xarr = weather.load_dataset()

# Prepare X
X_3d = weather.prepare_X_features(X_xarr, mode="3d") # (time, channels, lat*lon)
X_3d = weather.reshape_for_ml(X_3d)                  # (lat*lon, time, channels)

# Prepare y
y_3d = weather.prepare_y_targets(y_xarr, mode="3d")  # (vars, lat, lon)
y_2d = weather.flatten_y(y_3d)                       # (lat*lon, vars)

# Ready for ML
X_weather, y_weather = X_3d, y_2d
print(X_weather.shape, y_weather.shape)  # (2048, 92044, 3), (2048, 2)
print(f"X_weather memory: {X_weather.nbytes/1024**2:.2f} MB")


In [ ]:
"Camels DE"

camels_root       = "../public_datasets/3D/camels_de"
# camels_root       = "/Users/fouadabiad/Downloads/camels_de"
attributes_folder = camels_root
timeseries_folder = os.path.join(camels_root, "timeseries")
zarr_path         = os.path.join(camels_root, "camels_de_timeseries.zarr")

def load_X_from_scratch(timeseries_folder, zarr_path):
    # --- Delete Zarr if needed ---
    if os.path.exists(zarr_path):
        shutil.rmtree(zarr_path)

    # --- Load X ---
    ts_files  = sorted(glob(os.path.join(timeseries_folder, "*.csv")))
    ts_arrays = []

    for f in ts_files:
        df  = pd.read_csv(f, index_col=0)  # (time, catchments)
        df  = df.drop(columns=['date', 'discharge_vol_obs', 'discharge_spec_obs', 'water_level_obs'])
        arr = df.values
        ts_arrays.append(arr)

    # Stack along new axis -> (time, catchments, variables)
    X_np = np.stack(ts_arrays, axis=2)
    X_np = np.transpose(X_np, (2, 0, 1))
    print("X shape after transpose:", X_np.shape)  # (1582, 25568, 21)

    X_da = xr.DataArray(X_np, dims=("catchment", "time", "variable"))
    X_da.to_dataset(name="X").to_zarr(zarr_path, mode="w")
    print("Saved X to Zarr:", zarr_path)

    X_da_lazy = xr.open_zarr(zarr_path)["X"].values
    return X_da_lazy

def load_y_from_scratch(attributes_folder):
    """Also removed str cols"""
    attr_files = glob(os.path.join(attributes_folder, "CAMELS_DE_*.csv"))
    y_list     = [pd.read_csv(f, index_col=0) for f in attr_files]
    y_germany  = pd.concat(y_list, axis=1)
    y_germany = y_germany.select_dtypes(exclude='object').to_numpy()
    return y_germany

if os.path.exists(zarr_path):
    X_germany = xr.open_zarr(zarr_path)["X"].values
else:
    X_germany = load_X_from_scratch(timeseries_folder, zarr_path)
y_germany = load_y_from_scratch(attributes_folder)


print(f"X shape: {X_germany.shape} ({X_germany.nbytes / 1024**2:.2f} MB)")  # (1582, 25568, 21)
print(f"y shape: {y_germany.shape} ({y_germany.nbytes / 1024**2:.2f} MB)")  # (1582, 25568, 21)


In [ ]:
"""India dataset"""
data_path      = "../public_datasets/3D/india_catchments"
forcing_folder = "catchment_mean_forcings"
clim_file      = "attributes_csv/camels_ind_clim.csv"

# Load y (climate attributes)
y_df    = pd.read_csv(f"{data_path}/{clim_file}")
catchment_ids = y_df.iloc[:,0].astype(int).values
y_india = y_df.iloc[:, 1:]

# Load X (forcing time series)
forcing_files    = sorted(os.listdir(f"{data_path}/{forcing_folder}"))
X_list, file_ids = [], []

for f in forcing_files:
    path = os.path.join(f"{data_path}/{forcing_folder}", f)
    df   = pd.read_csv(path)
    X_list.append(df.drop(columns=['year','month','day','pet(mm/day)']).values)
    file_ids.append(int(f.split('.')[0]))

X       = np.stack(X_list, axis=0)
order   = [file_ids.index(cid) for cid in catchment_ids]
X_india = X[order]

print("X_india shape:", X_india.shape, "y_india shape:", y_india.shape)


In [ ]:
"""China Weather 2k data"""

dataset_location = "../public_datasets/3D/china_weather/weather2k.npy"
# dataset_location = "/Users/fouadabiad/Downloads/weather2k.npy"
china_data       = np.load(dataset_location, mmap_mode='r')  # read-only memory map
china_data       = china_data.transpose(0,2,1) # (stations, timesteps, features)

y_indices = [4, 5, 6, 7, 10] # [T, mnt, mxt, rh, ws], to remove
y         = china_data[:, -1, y_indices] # last timestep
mask      = np.ones(china_data.shape[2], dtype=bool)
mask[y_indices] = False
X         = china_data[:, :, mask]

print(f"X shape: {X.shape} ({X.nbytes / 1024**2:.2f} MB)")  # (patients, conditions)
print(f"y shape: {y.shape} ({y.nbytes / 1024**2:.2f} MB)")  # (patients, conditions)


In [ ]:
"""[to remove] Preprocess dataset"""

def downsample_pages_and_rows(X: np.ndarray, y: np.ndarray, page_frac: float = 0.1,
                              row_frac: float = 0.4) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Downsample pages and rows from a 3D dataset with uniform sampling
        X: Input data of shape (n_pages, n_rows, n_features).
        y: Target values of shape (n_pages, ...) corresponding to pages.
        page_frac: Fraction of pages to sample (default 0.1).
        row_frac: Fraction of rows to sample (default 0.4).
    Returns:
        X_small: Downsampled data of shape (num_pages, num_rows, n_features).
        y_small: Target values corresponding to sampled pages.
        page_idx: Indices of selected pages.
        row_idx: Indices of selected rows.
    Notes:
        - Pages and rows are sampled uniformly (evenly spaced).
        - Sampling is deterministic; no randomness is used."""

    n_pages, n_rows, n_features = X.shape
    num_pages = max(1, int(n_pages * page_frac))
    num_rows  = max(1, int(n_rows * row_frac))

    # Uniform page sampling
    page_idx = np.linspace(0, n_pages - 1, num_pages, dtype=int)

    # Uniform row sampling
    row_idx = np.linspace(0, n_rows - 1, num_rows, dtype=int)

    X_small = X[page_idx][:, row_idx, :]
    y_small = y[page_idx]

    return X_small, y_small, page_idx, row_idx

X_small, y_small, _, _ = downsample_pages_and_rows(X, y.values, page_frac=0.1, row_frac=0.1)

# 5. Train/test split
y_train, y_test = train_test_split(y_small, test_size=0.2, random_state=42, shuffle=False) # only split y (no random shuffle) to align with timeVAE code

# 6. Apply target encoding to categorical columns
cat_cols = y.select_dtypes(include=['object','category']).columns

def apply_target_encoding(cat_cols, y_train, y_test):
    if len(cat_cols) > 0:
        idx         = [y.columns.get_loc(c) for c in cat_cols]
        y_train_df  = pd.DataFrame(y_train[:, idx], columns=cat_cols)
        y_test_df   = pd.DataFrame(y_test[:, idx],  columns=cat_cols)
        encoder     = ce.TargetEncoder(cols=cat_cols)
        y_train_enc = encoder.fit_transform(y_train_df, y_train[:,0])
        y_test_enc  = encoder.transform(y_test_df)
        y_train[:, idx] = y_train_enc.values
        y_test[:, idx]  = y_test_enc.values
    return y_train.astype(float), y_test.astype(float)

y_train, y_test = apply_target_encoding(cat_cols, y_train, y_test)

# Ensure 2D
y_train = np.atleast_2d(y_train)
y_test  = np.atleast_2d(y_test)

# 7. Scale X
# X_mean = X_train.mean(axis=(0,1))
# X_std  = X_train.std(axis=(0,1))
# X_std[X_std==0] = 1.0
# X_train_scaled  = (X_train - X_mean)/X_std
# X_test_scaled   = (X_test - X_mean)/X_std

# 8. Scale y
y_mean = y_train.mean(axis=0)
y_std  = y_train.std(axis=0)
y_std[y_std==0] = 1.0
y_train_scaled  = (y_train - y_mean)/y_std
y_test_scaled   = (y_test - y_mean)/y_std

print(f"y_small shape: {y_small.shape}")
print(f"X_small shape: {X_small.shape}, X_small memory: {X_small.nbytes/1024**2:.2f} MB")
print(f"y_train_scaled shape: {y_train_scaled.shape}, y_test_scaled shape: {y_test_scaled.shape}")


In [2]:
"""Preprocess dataset, as class"""
from dataset_loaders import ECGLoader
import category_encoders as ce

class DatasetPreprocessor:
    """Preprocess datasets: downsample, train/test split, categorical encoding, and scaling."""
    
    def __init__(self, page_frac=0.1, row_frac=0.4, test_size=0.2, scale_X=True):
        self.page_frac = page_frac
        self.row_frac  = row_frac
        self.test_size = test_size
        self.scale_X   = scale_X
        self.y_mean    = None
        self.y_std     = None
        self.X_scaler  = None

    def fit_transform(self, X: np.ndarray, y: np.ndarray, split_X: bool = True) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """Downsample, split, encode, and scale dataset. X always stays 3D."""
        self._prepare_targets(y)
        X_small, y_small = self._downsample_pages_and_rows(X, self.y_df)

        # Split targets
        y_train, y_test = train_test_split(y_small, test_size=self.test_size, shuffle=False)
        y_train, y_test = self._encode_categorical(y_train, y_test)
        y_train_scaled, y_test_scaled = self._scale_targets(y_train, y_test)

        if not split_X:
            # Return full downsampled 3D X
            # X_small = X_small.astype(np.float16)
            return X_small, y_train_scaled, y_test_scaled, None

        # Split X along first axis (pages) without flattening
        X_train, X_test = train_test_split(X_small, test_size=self.test_size, shuffle=False)

        if self.scale_X:
            # Optionally scale while keeping 3D shape
            ns, nr, nf = X_train.shape
            ns_test, nr_test, nf_test = X_test.shape

            # Flatten temporarily for StandardScaler
            X_train_flat   = X_train.reshape(ns, -1)
            X_test_flat    = X_test.reshape(ns_test, -1)
            self.X_scaler  = StandardScaler()
            X_train_scaled = self.X_scaler.fit_transform(X_train_flat).reshape(ns, nr, nf)
            X_test_scaled  = self.X_scaler.transform(X_test_flat).reshape(ns_test, nr_test, nf_test)

            # <--- cast to float16 after scaling --->
            # X_train_scaled = X_train_scaled.astype(np.float16)
            # X_test_scaled  = X_test_scaled.astype(np.float16)

            return X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled
        else:
            return X_train, X_test, y_train_scaled, y_test_scaled

    def _prepare_targets(self, y):
        if isinstance(y, np.ndarray):
            self.y_df         = pd.DataFrame(y)
            self.cat_cols     = []
            self.numeric_cols = self.y_df.columns.tolist()
        else:
            self.y_df         = y.copy()
            self.numeric_cols = self.y_df.select_dtypes(include=[np.number]).columns.tolist()
            self.cat_cols     = self.y_df.select_dtypes(include=['object','category']).columns.tolist()

    def _downsample_pages_and_rows(self, X, y_df):
        n_pages, n_rows, _ = X.shape
        num_pages = max(1, int(n_pages * self.page_frac))
        num_rows  = max(1, int(n_rows * self.row_frac))
        page_idx  = np.linspace(0, n_pages-1, num_pages, dtype=int)
        row_idx   = np.linspace(0, n_rows-1, num_rows, dtype=int)
        X_small   = X[page_idx][:, row_idx, :]
        y_small   = y_df.iloc[page_idx].values if isinstance(y_df, pd.DataFrame) else y_df[page_idx]
        return X_small, y_small

    def _encode_categorical(self, y_train, y_test):
        if len(self.cat_cols) > 0:
            idx         = [self.y_df.columns.get_loc(c) for c in self.cat_cols]
            y_train_df  = pd.DataFrame(y_train[:, idx], columns=self.cat_cols)
            y_test_df   = pd.DataFrame(y_test[:, idx], columns=self.cat_cols)
            encoder     = ce.TargetEncoder(cols=self.cat_cols)
            y_train_enc = encoder.fit_transform(y_train_df, y_train[:,0])
            y_test_enc  = encoder.transform(y_test_df)
            y_train[:, idx] = y_train_enc.values
            y_test[:, idx]  = y_test_enc.values
        return y_train.astype(float), y_test.astype(float)

    def _scale_targets(self, y_train, y_test):
        self.y_mean    = y_train.mean(axis=0)
        self.y_std     = y_train.std(axis=0)
        self.y_std[self.y_std == 0] = 1.0
        y_train_scaled = (y_train - self.y_mean) / self.y_std
        y_test_scaled  = (y_test - self.y_mean) / self.y_std
        y_train_scaled = np.atleast_2d(y_train_scaled)
        y_test_scaled  = np.atleast_2d(y_test_scaled)
        return y_train_scaled, y_test_scaled

def load_and_preprocess_dataset(desired_dataset: str, split_X: bool = False, segment_duration_sec: int = 200,
                                max_records: int = 2000) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Load a public 3D dataset, downsample, preprocess, and optionally flatten/scale X."""
    
    # ----- Load X, y depending on dataset -----
    if desired_dataset == "ecg":
        ECG_data_path = "../public_datasets/3D/ptb-xl-1.0.3"
        loader        = ECGLoader(ECG_data_path)
        X, y, _, _    = loader.load_dataset(sampling="lr", target="diagnostic_superclass_multi",
                                             segment_duration_sec=segment_duration_sec,
                                             max_records=max_records, continuous_target=True)
    elif desired_dataset == "argoverse":
        argoverse_data_path = "../public_datasets/3D/argoverse_forecasting"
        folder_name = "00a0ec58-1fb9-4a2b-bfd7-f4e5da7a9eff"
        file_name   = "scenario_00a0ec58-1fb9-4a2b-bfd7-f4e5da7a9eff.parquet"
        X, y        = process_argoverse_parquet(f"{argoverse_data_path}/{folder_name}/{file_name}")

    elif desired_dataset == "weather":
        dataset_folder = "../public_datasets/3D/weather_bench"
        variables_X    = ["2m_temperature", "10m_u_component_of_wind", "10m_v_component_of_wind"]
        variables_y    = ["mean_sea_level_pressure", "total_precipitation_6hr"]
        weather        = WeatherDataset(dataset_folder, variables_X, variables_y)
        X_xarr, y_xarr = weather.load_dataset()
        X_3d = weather.prepare_X_features(X_xarr, mode="3d")
        X_3d = weather.reshape_for_ml(X_3d)
        y_3d = weather.prepare_y_targets(y_xarr, mode="3d")
        y_2d = weather.flatten_y(y_3d)
        X, y = X_3d, y_2d

    elif desired_dataset == "india_catchment":
        data_path        = "../public_datasets/3D/india_catchments"
        forcing_folder   = "catchment_mean_forcings"
        clim_file        = "attributes_csv/camels_ind_clim.csv"

        y_df             = pd.read_csv(f"{data_path}/{clim_file}")
        catchment_ids    = y_df.iloc[:,0].astype(int).values
        y                = y_df.iloc[:, 1:]
        forcing_files    = sorted(os.listdir(f"{data_path}/{forcing_folder}"))
        X_list, file_ids = [], []
        for f in forcing_files:
            df = pd.read_csv(os.path.join(data_path, forcing_folder, f))
            X_list.append(df.drop(columns=['year','month','day','pet(mm/day)']).values)
            file_ids.append(int(f.split('.')[0]))

        X     = np.stack(X_list, axis=0)
        order = [file_ids.index(cid) for cid in catchment_ids]
        X     = X[order]
    
    elif desired_dataset == "germany_catchment":
        if os.path.exists(zarr_path):
            X = xr.open_zarr(zarr_path)["X"].values
        else:
            X = load_X_from_scratch(timeseries_folder, zarr_path)

        y = load_y_from_scratch(attributes_folder)

        if np.isnan(X).any():
            X = np.nan_to_num(X)
        if np.isnan(y).any():
            y = np.nan_to_num(y)

    elif desired_dataset == "china_weather":
        dataset_location = "../public_datasets/3D/china_weather" #"/Users/fouadabiad/Downloads/weather2k.npy"
        file_name        = "weather2k.npy"
        china_data       = np.load(os.path.join(dataset_location, file_name), mmap_mode='r')  # read-only memory map
        china_data       = china_data.transpose(0,2,1) # (stations, timesteps, features)

        y_indices = [4, 5, 6, 7, 10] # [T, mnt, mxt, rh, ws], to remove
        y         = china_data[:, -1, y_indices] # last timestep
        mask      = np.ones(china_data.shape[2], dtype=bool)
        mask[y_indices] = False
        X         = china_data[:, :, mask]

    else:
        raise ValueError(f"Unknown dataset: {desired_dataset}")

    # ----- Downsample + preprocess -----
    preprocessor = DatasetPreprocessor(page_frac=downsampling_dict[desired_dataset][0],
                                       row_frac=downsampling_dict[desired_dataset][1], test_size=0.2)
    result = preprocessor.fit_transform(X, y, split_X=split_X)
    return result

    # ----- Cast to float16 for memory -----
    # if split_X:
    #     X_train, X_test, y_train_scaled, y_test_scaled = result
    #     X_train = X_train.astype(np.float16)
    #     return X_train, X_test, y_train_scaled, y_test_scaled
    # else:
    #     X_small, y_train_scaled, y_test_scaled, _ = result
    #     X_small = X_small.astype(np.float16)
    #     return X_small, y_train_scaled, y_test_scaled, None


def load_or_preprocess(desired_dataset: str, segment_duration_sec: int = 150,
                       max_records: int = 15000) -> tuple[np.ndarray, ...]:
    """Load cached preprocessed dataset if available, otherwise preprocess and cache it."""
    page_frac, row_frac = downsampling_dict[desired_dataset]
    new_dir_name        = f"{desired_dataset}_{page_frac}_{row_frac}"
    save_dir            = f"../interim_data/{new_dir_name}"
    
    if os.path.exists(save_dir): # Load from cache
        print("Path exists, loading from cache:", save_dir)
        X_train        = np.load(f"{save_dir}/X_train.npz")['data']
        X_test         = np.load(f"{save_dir}/X_test.npz")['data']
        y_train_scaled = np.load(f"{save_dir}/y_train.npz")['data']
        y_test_scaled  = np.load(f"{save_dir}/y_test.npz")['data']

        X_small        = np.load(f"{save_dir}/X_small.npz")['data']
        y_train_small  = np.load(f"{save_dir}/y_train_small.npz")['data']
        y_test_small   = np.load(f"{save_dir}/y_test_small.npz")['data']
    else: # Preprocess fresh
        print("Path does not exist, preprocessing fresh:", save_dir)
        os.makedirs(save_dir, exist_ok=True)

        X_train, X_test, y_train_scaled, y_test_scaled = load_and_preprocess_dataset(
            desired_dataset, split_X=True, segment_duration_sec=segment_duration_sec, 
            max_records=max_records)

        # Also keep a smaller version (for timevae)
        X_small, y_train_small, y_test_small, _ = load_and_preprocess_dataset(
            desired_dataset, split_X=False, segment_duration_sec=segment_duration_sec, 
            max_records=max_records)

        # Save to cache
        np.savez_compressed(f"{save_dir}/X_train.npz", data=X_train.astype(np.float32))
        np.savez_compressed(f"{save_dir}/X_test.npz", data=X_test.astype(np.float32))
        np.savez_compressed(f"{save_dir}/y_train.npz", data=y_train_scaled.astype(np.float32))
        np.savez_compressed(f"{save_dir}/y_test.npz", data=y_test_scaled.astype(np.float32))
        np.savez_compressed(f"{save_dir}/X_small.npz", data=X_small.astype(np.float32))
        np.savez_compressed(f"{save_dir}/y_train_small.npz", data=y_train_small.astype(np.float32))
        np.savez_compressed(f"{save_dir}/y_test_small.npz", data=y_test_small.astype(np.float32))
    return X_train, X_test, y_train_scaled, y_test_scaled, X_small, y_train_small, y_test_small

downsampling_dict = {"ecg": (0.05, 0.08),
                     "argoverse": (0.3, 0.1),
                     "weather": (0.3, 0.01),
                     "india_catchment": (0.3, 0.08),
                     "germany_catchment": (0.2, 0.02),
                     "china_weather": (0.25, 0.05)}

desired_dataset = "germany_catchment" # options: ecg, argoverse, weather, india_catchment, germany_catchment, china_weather
baseline        = "timevae"  # options: "timevae", "ts2vec", "moment", "?", "?"

X_train, X_test, y_train_scaled, y_test_scaled, X_small, y_train_small, y_test_small = load_or_preprocess(desired_dataset)

# X_train = X_train.astype(np.float16)
# X_test  = X_test.astype(np.float16)
# X_small = X_small.astype(np.float16)
print(f"X_train: {X_train.shape} ({X_train.nbytes/1024**2:.2f} MB), X_test: {X_test.shape} ({X_test.nbytes/1024**2:.2f} MB)")
print(f"X_small: {X_small.shape} ({X_small.nbytes/1024**2:.2f} MB)")

page_frac, row_frac = downsampling_dict[desired_dataset]

if baseline == "timevae":
    os.makedirs('../interim_data', exist_ok=True)
    np.savez_compressed(f"../../timeVAE/data/{desired_dataset}_{page_frac}_{row_frac}.npz", data=np.array(X_small, dtype=np.float32))
    print(f"Saved {desired_dataset}_{page_frac}_{row_frac} to TimeVAE repo")


Path exists, loading from cache: ../interim_data/germany_catchment_0.2_0.02
X_train: (252, 511, 21) (10.32 MB), X_test: (64, 511, 21) (2.62 MB)
X_small: (316, 511, 21) (12.94 MB)
Saved germany_catchment_0.2_0.02 to TimeVAE repo


In [ ]:
"General Prediction"

class Preds():
    "Class of predictors to predict y from X"

    @staticmethod
    def predict_linreg(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray) -> float:
        """Train linear predictor"""
        model  = LinearRegression()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        return root_mean_squared_error(y_test, y_pred)

    @staticmethod
    def predict_catboost_multioutput(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray) -> Tuple[Optional[MultiOutputRegressor], np.ndarray, float]:
        """Train multi-output CatBoost models and predict test set.
        Returns:
            model: trained MultiOutputRegressor (or None if all targets constant)
            y_pred: predictions on test set
            rmse: RMSE across all targets"""
        y_pred           = np.zeros_like(y_test, dtype=float)
        non_constant_idx = [i for i in range(y_train.shape[1])
                            if not np.all(y_train[:, i] == y_train[0, i])]
        if non_constant_idx:
            model = MultiOutputRegressor(CatBoostRegressor(iterations=500, learning_rate=0.1, depth=4, verbose=0))
            model.fit(X_train, y_train[:, non_constant_idx])
            y_pred[:, non_constant_idx] = model.predict(X_test)
            for i in range(y_train.shape[1]):
                if np.all(y_train[:, i] == y_train[0, i]):
                    y_pred[:, i] = y_train[0, i]
        else:
            model = None
        rmse = root_mean_squared_error(y_test, y_pred)
        return model, y_pred, rmse

    @staticmethod
    def cluster_and_label(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray,
                        y_test: np.ndarray, n_clusters: int = 10, random_state: int = 42) -> float:
        """Cluster train features with KMeans, assign representative y (mean per cluster), predict test labels by cluster assignment,
        and compute RMSE. this is considered unsupervised, as the clustering happens to X only
        - X_train: Training features (N_train, D).
        - y_train: Training targets (N_train, T).
        - X_test: Test features (N_test, D).
        - y_test: Test targets (N_test, T).
        - n_clusters: Number of KMeans clusters.
        - random_state: Random seed for reproducibility.
        Returns: Mean squared error on test set."""
        kmeans         = KMeans(n_clusters=n_clusters, random_state=random_state)
        train_clusters = kmeans.fit_predict(X_train)

        # mean target vector per cluster
        y_cluster     = {cluster: y_train[train_clusters == cluster].mean(axis=0)
                        for cluster in range(n_clusters)}
        test_clusters = kmeans.predict(X_test)
        y_pred        = np.stack([y_cluster[cluster] for cluster in test_clusters], axis=0)
        return root_mean_squared_error(y_test, y_pred)

    @staticmethod
    def predict_rf_multioutput(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray) -> float:
        """Train multi-output Random Forest and compute RMSE."""
        model  = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        return root_mean_squared_error(y_test, y_pred)

    @staticmethod
    def predict_elasticnet_multioutput(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray,
                                    alpha: float = 0.1, l1_ratio: float = 0.5) -> Tuple[MultiOutputRegressor, np.ndarray, float]:
        """Train multi-output ElasticNet (L1+L2) and predict test set.
        Returns:
            model: trained MultiOutputRegressor
            y_pred: predictions on test set
            rmse: root mean squared error"""
        model  = MultiOutputRegressor(ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=1000, random_state=42))
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        rmse   = root_mean_squared_error(y_test, y_pred)
        return model, y_pred, rmse

    @staticmethod
    def evaluate_models_on_dataset(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray):
        """Evaluate various models on the dataset and print RMSE results."""
        linreg_loss       = Preds.predict_linreg(X_train, y_train, X_test, y_test)
        _, _, catboost_loss = Preds.predict_catboost_multioutput(X_train, y_train, X_test, y_test)
        unsupervised_rmse = Preds.cluster_and_label(X_train, y_train, X_test, y_test, n_clusters=5)
        rf_rmse           = Preds.predict_rf_multioutput(X_train, y_train, X_test, y_test)
        _, _, el_rmse     = Preds.predict_elasticnet_multioutput(X_train, y_train, X_test, y_test, alpha=0.1, l1_ratio=0.5)
        return linreg_loss, catboost_loss, unsupervised_rmse, rf_rmse, el_rmse

print(f"dataset: {desired_dataset}_{page_frac}_{row_frac}")
print( "    RMSE    | LinReg | CatBoost | Cluster | RForest | ElasticNet")

# ====== X mean ======
X_train_2d = X_train.mean(axis=1).astype(np.float32)
X_test_2d  = X_test.mean(axis=1).astype(np.float32)
linreg_loss, catboost_loss, unsupervised_rmse, rf_rmse, el_rmse = Preds.evaluate_models_on_dataset(X_train_2d, y_train_scaled,
                                                                                                    X_test_2d, y_test_scaled)
print(f"& mean(X)   & {linreg_loss:.4f} & {catboost_loss:.4f}   & {unsupervised_rmse:.4f}  & {rf_rmse:.4f}  & {el_rmse:.4f} & - \\")

# ====== X last ======
# X_train_2d = X_train[:, -10:, :].mean(axis=1).astype(np.float32)
# X_test_2d  = X_test[:, -10:, :].mean(axis=1).astype(np.float32)
# linreg_loss, catboost_loss, unsupervised_rmse, rf_rmse, el_rmse = Predictors.evaluate_models_on_dataset(X_train_2d, y_train_scaled,
#                                                                                              X_test_2d, y_test_scaled)
# print(f" last(X)   | {linreg_loss:.4f} | {catboost_loss:.4f}   | {unsupervised_rmse:.3f}   | {rf_rmse:.3f}   | {el_rmse:.3f}")

# ====== X random ======
n_train, rows, _ = X_train.shape
n_test     = X_test.shape[0]
train_idx  = np.random.randint(0, rows, size=n_train)
test_idx   = np.random.randint(0, rows, size=n_test)
X_train_2d = X_train[np.arange(n_train), train_idx, :].astype(np.float32)
X_test_2d  = X_test[np.arange(n_test), test_idx, :].astype(np.float32)
linreg_loss, catboost_loss, unsupervised_rmse, rf_rmse, el_rmse = Preds.evaluate_models_on_dataset(X_train_2d, y_train_scaled,
                                                                                                   X_test_2d, y_test_scaled)
print(f"& random(X) & {linreg_loss:.4f} & {catboost_loss:.4f}   & {unsupervised_rmse:.4f}  & {rf_rmse:.4f}  & {el_rmse:.4f} & - \\ ")



In [ ]:
"""TimeVAE"""
z_train     = np.load(f"../interim_data/z_train_{desired_dataset}.npy")
z_valid     = np.load(f"../interim_data/z_valid_{desired_dataset}.npy")
z_test      = np.load(f"../interim_data/z_test_{desired_dataset}.npy")
z_train_val = np.concatenate([z_train, z_valid], axis=0) #concat z_train and val to = y shape
print(f"z_train: {z_train.shape}, z_valid: {z_valid.shape}, z_train_val: {z_train_val.shape}, z_test: {z_test.shape}")
print(f"y_train_scaled: {y_train_scaled.shape}, y_test_scaled: {y_test_scaled.shape}")

# Train NN predictor
predictor_lr      = 0.006
predictor_epochs  = 200
predictor_dropout = 0.05
predictor_hidden_sizes = [32, 48, 64] # latent to y output

nn_predictor = MLPHead(input_dim=z_train.shape[1], output_dim=y_train_scaled.shape[1],
                           hidden_sizes=predictor_hidden_sizes, lr=predictor_lr,
                           epochs=predictor_epochs, dropout=predictor_dropout, device=device)
nn_predictor.train(z_train_val, y_train_scaled, z_test, y_test_scaled)
nn_rmse = nn_predictor.evaluate(z_test, y_test_scaled)

# ====== plain predictors ======
linreg_loss, catboost_loss, unsupervised_rmse, rf_rmse, el_rmse = Preds.evaluate_models_on_dataset(z_train_val, y_train_scaled,
                                                                                                   z_test, y_test_scaled)
print(f"dataset: {desired_dataset}, method: timevae")
print( "    RMSE      | LinReg | CatBoost | Cluster | RForest | ElasticNet | NN")
print(f"& Z (timevae) & {linreg_loss:.4f} & {catboost_loss:.4f}   & {unsupervised_rmse:.4f}  & {rf_rmse:.4f}  & {el_rmse:.4f} \
    & {nn_rmse:.4f} \\ ")


In [ ]:
"""TS2Vec"""
# from ts2vec import TS2Vec

# class TS2VecEncoder:
#     """TS2Vec + Linear Predictor pipeline with externally set hyperparameters
#         1. split_scale(X)
#         2. fit_ts2vec(X_train, X_test)
#         3. convert_to_tensors(z_train, z_test, y_train, y_test)
#         4. make_predictor(latent_dim, y_dim)
#         5. train_predictor(...)
#         6. evaluate(...)"""

#     def __init__(self, z_pooling: str = "mean", device="cpu"):
#         self.z_pooling = z_pooling
#         self.ts_model  = None
#         self.device    = device

#     def fit(self, X_train, hidden_dims=12, output_dims=8, depth=5, batch_size=16, n_epochs=20):
#         """Fit TS2Vec on training data (unsupervised)."""
#         X_train = X_train.astype(np.float32)
#         self.ts_model = TS2Vec(input_dims=X_train.shape[2],
#                                hidden_dims=hidden_dims,
#                                depth=depth,
#                                output_dims=output_dims,
#                                batch_size=batch_size,
#                                device=self.device)
#         self.ts_model.fit(X_train, n_epochs=n_epochs, verbose=True)

#     def encode(self, X, pooling=None):
#         """Encode X into latent embeddings with pooling."""
#         if self.ts_model is None:
#             raise ValueError("TS2Vec model not trained. Call fit first.")
#         z = self.ts_model.encode(X.astype(np.float32))
#         p = pooling or self.z_pooling
#         if p == "mean":
#             return z.mean(axis=1)
#         elif p == "max":
#             return z.max(axis=1)
#         elif p == "last":
#             return z[:, -1, :]
#         else:
#             raise ValueError(f"Unknown pooling: {p}")

z_pooling_method   = "mean"
ts2vec_hidden_dims = 16 # units in each layer (> than latent dim)
ts2vec_output_dims = 8 # latent dim
ts2vec_depth       = 2 # num layers
ts2vec_batch_size  = 32
ts2vec_epochs      = 20

predictor_lr       = 0.009
predictor_epochs   = 150
predictor_dropout  = 0.05
predictor_hidden_sizes = [32, 48, 64] # latent to y output

# 1️⃣ Fit TS2Vec
ts2vec_encoder = TS2VecEncoder(z_pooling=z_pooling_method, device=device)
ts2vec_encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_output_dims,
                   depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=ts2vec_epochs)
z_train = ts2vec_encoder.encode(X_train)
z_test  = ts2vec_encoder.encode(X_test)

# 2️⃣ Train NN predictor
nn_predictor = MLPHead(input_dim=z_train.shape[1], output_dim=y_train_scaled.shape[1],
                           hidden_sizes=predictor_hidden_sizes, lr=predictor_lr,
                           epochs=predictor_epochs, dropout=predictor_dropout, device=device)
nn_predictor.train(z_train, y_train_scaled, z_test, y_test_scaled)
nn_rmse = nn_predictor.evaluate(z_test, y_test_scaled)


# ====== plain predictors ======
linreg_loss, catboost_loss, unsupervised_rmse, rf_rmse, el_rmse = Preds.evaluate_models_on_dataset(z_train, y_train_scaled,
                                                                                                   z_test, y_test_scaled)
print(f"dataset: {desired_dataset}, method: ts2vec")
print( "    RMSE   | LinReg | CatBoost | Cluster | RForest | ElasticNet | NN")
print(f"& Z (ts2vec) & {linreg_loss:.4f} & {catboost_loss:.4f} & {unsupervised_rmse:.4f}  & {rf_rmse:.4f}  & {el_rmse:.4f} \
    & {nn_rmse:.4f} \\ ")


Epoch #0: loss=3.5992997714451382
Epoch #1: loss=3.845799514225551
Epoch #2: loss=3.6604390144348145
Epoch #3: loss=3.6499596663883755
Epoch #4: loss=3.5915571621486118


KeyboardInterrupt: 

In [ ]:
"""MOMENT (uses MOMENT-1-large model)"""
from momentfm import MOMENTPipeline

model_type   = "repres_learning" # options: classification (= regression), repres_learning
model_name   = "MOMENT-1-base" #"MOMENT-1-large"
reload_model = True

if reload_model == True:
    if model_type == "classification": # classification/regression
        classes_in_y = len(np.unique(y_train_scaled))
        moment_model = MOMENTPipeline.from_pretrained(f"AutonLab/{model_name}", 
                                                      model_kwargs={'task_name': 'classification',
                                                                    'n_channels': X_train.shape[2], 'num_class': classes_in_y},)
    elif model_type == "repres_learning": # representation learning
        moment_model = MOMENTPipeline.from_pretrained(f"AutonLab/{model_name}", 
                                                      model_kwargs={"task_name": "embedding"},)
    print(f"Doing {model_type} with {model_name}!")
    moment_model.init()
    moment_model.eval()

# 1️⃣ Encode train and test separately (NOTE: model expects data as [batch, channels, seq_len] )
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).permute(0, 2, 1)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32).permute(0, 2, 1)

with torch.no_grad():
    train_outputs = moment_model(x_enc=X_train_tensor)
    test_outputs  = moment_model(x_enc=X_test_tensor)

z_train = train_outputs.embeddings.cpu().numpy()
z_test  = test_outputs.embeddings.cpu().numpy()

if model_type == "classification":
    z_train = z_train.mean(axis=1)  # or max(axis=1)
    z_test  = z_test.mean(axis=1)

print(X_train.shape, z_train.shape)

# 2️⃣ Train downstream NN predictor
predictor_lr      = 0.009
predictor_epochs  = 100
predictor_dropout = 0.05
predictor_hidden_sizes = [32, 48, 64] # latent to y output

nn_predictor = MLPHead(input_dim=z_train.shape[1], output_dim=y_train_scaled.shape[1],
                           hidden_sizes=predictor_hidden_sizes, lr=predictor_lr,
                           epochs=predictor_epochs, dropout=predictor_dropout, device=device)
nn_predictor.train(z_train, y_train_scaled, z_test, y_test_scaled)
nn_rmse = nn_predictor.evaluate(z_test, y_test_scaled)

# ====== plain predictors ======
linreg_loss, catboost_loss, unsupervised_rmse, rf_rmse, el_rmse = Preds.evaluate_models_on_dataset(z_train, y_train_scaled,
                                                                                                   z_test, y_test_scaled)
print(f"dataset: {desired_dataset}, method: moment")
print( "    RMSE   | LinReg | CatBoost | Cluster | RForest | ElasticNet | NN")
print(f"& Z ({model_type}) & {linreg_loss:.4f} & {catboost_loss:.4f} & {unsupervised_rmse:.4f}  & {rf_rmse:.4f}  & {el_rmse:.4f} \
    & {nn_rmse:.4f} \\ ")


In [ ]:
"shortcut from waiting for moment to finish running"
from sklearn.decomposition import PCA
from sklearn.random_projection import GaussianRandomProjection

linreg_loss       = Preds.predict_linreg(z_train, y_train_scaled, z_test, y_test_scaled)
print("linreg", linreg_loss)

def predict_catboost_multioutput(
    X_train: np.ndarray, y_train: np.ndarray,
    X_test: np.ndarray, y_test: np.ndarray
) -> Tuple[Optional[MultiOutputRegressor], np.ndarray, float]:
    """
    Train multi-output CatBoost with random projection for speed and predict test set.
    Handles constant targets correctly.
    
    Returns:
        model: trained MultiOutputRegressor (or None if all targets constant)
        y_pred: predictions on test set
        rmse: RMSE across all targets
    """
    y_pred = np.zeros_like(y_test, dtype=float)
    
    # Step 0: Identify non-constant targets
    non_constant_idx = [i for i in range(y_train.shape[1])
                        if not np.all(y_train[:, i] == y_train[0, i])]
    
    if non_constant_idx:
        # Step 1: Fast dimensionality reduction
        rp = GaussianRandomProjection(n_components=2048, random_state=42)
        X_train_rp = rp.fit_transform(X_train)
        X_test_rp  = rp.transform(X_test)
        
        # Step 2: Train minimal CatBoost
        model = MultiOutputRegressor(
            CatBoostRegressor(
                iterations=100,        # minimal for speed
                depth=4,             # shallow tree
                learning_rate=0.1,
                verbose=0,
                thread_count=-1      # all CPU cores
            ),
            n_jobs=-1               # parallel outputs
        )
        # iterations=500, learning_rate=0.1, depth=4

        model.fit(X_train_rp, y_train[:, non_constant_idx])
        y_pred[:, non_constant_idx] = model.predict(X_test_rp)
        
        # Step 3: Fill constant outputs
        for i in range(y_train.shape[1]):
            if i not in non_constant_idx:
                y_pred[:, i] = y_train[0, i]
    else:
        model = None
    
    rmse = root_mean_squared_error(y_test, y_pred)
    return model, y_pred, rmse

model, y_pred, rmse = predict_catboost_multioutput(z_train, y_train_scaled, z_test, y_test_scaled)
print("RMSE:", rmse)

unsupervised_rmse = Preds.cluster_and_label(z_train, y_train_scaled, z_test, y_test_scaled, n_clusters=5)
print("cluster",unsupervised_rmse)

# rf_rmse           = Preds.predict_rf_multioutput(z_train, y_train_scaled, z_test, y_test_scaled)
print("rf",rf_rmse)
model  = MultiOutputRegressor(RandomForestRegressor(n_estimators=30, random_state=42, n_jobs=-1))
model = MultiOutputRegressor(RandomForestRegressor(
        n_estimators=10,    # reduced from 100 to speed up
        max_depth=15,       # limit tree depth to speed up
        min_samples_leaf=2, # prevents overfitting / speeds up slightly
        random_state=42, n_jobs=-1))
model.fit(z_train, y_train_scaled)
y_pred = model.predict(z_test)
rf_rmse = root_mean_squared_error(y_test_scaled, y_pred)
print("rf",rf_rmse)

_, _, el_rmse     = Preds.predict_elasticnet_multioutput(z_train, y_train_scaled, z_test, y_test_scaled, alpha=0.1, l1_ratio=0.5)
print("el", el_rmse)


# predictor_lr      = 0.008
# predictor_epochs  = 200
# predictor_dropout = 0.05
# predictor_hidden_sizes = [256, 128] # latent to y output

# nn_predictor = MLPHead(input_dim=z_train.shape[1], output_dim=y_train_scaled.shape[1],
#                            hidden_sizes=predictor_hidden_sizes, lr=predictor_lr,
#                            epochs=predictor_epochs, dropout=predictor_dropout, device=device)
# nn_predictor.train(z_train, y_train_scaled, z_test, y_test_scaled)
# nn_rmse = nn_predictor.evaluate(z_test, y_test_scaled)
# print("NN:", nn_rmse)

In [28]:
"functions for custom architecture"
import copy
from predictions import Decoder
from other_encoders.ts2vec_encoder import TS2VecEncoder

def make_augmentations(X: torch.Tensor,  augment_type: str, device, scale_factor: float = 0.1) -> torch.Tensor:
    """Apply a chosen augmentation to a 3D time series batch.
    - X: Input array of shape (batch, time, channels).
    - augment_type: Type of augmentation to apply.
    - scale_factor: Magnitude factor controlling strength of augmentation.
    returns augmented array with the same shape as X (except cropping)"""
    batches, timesteps, cols = X.shape
    X = X.to(device)

    if augment_type == "jitter":
        noise = torch.randn_like(X) * float(scale_factor)
        return X + noise
    if augment_type == "scaling":
        # per-sample, per-channel scaling factor
        factor = torch.randn(batches, 1, cols, device=device) * scale_factor + 1.0
        return X * factor
    if augment_type == "mag_warp":
        mag = torch.empty(batches, 1, 1, device=device).uniform_(1 - scale_factor, 1 + scale_factor)
        return X * mag
    if augment_type == "time_warp":
        # simple implementation: random small shifts via linear interpolation per sample/channel
        X_aug = torch.empty_like(X)
        for b in range(batches):
            # produce monotonic warp positions
            steps = torch.randn(timesteps, device=device) * scale_factor
            warp = torch.arange(timesteps, device=device) + torch.cumsum(steps, dim=0)
            warp = (warp - warp.min()) / (warp.max() - warp.min()) * (timesteps - 1)
            grid = warp.cpu().numpy()
            for c in range(cols):
                X_aug[b, :, c] = torch.from_numpy(
                    np.interp(np.arange(timesteps), grid, X[b, :, c].cpu().numpy())
                ).to(device)
        return X_aug
    if augment_type == "permutation":
        X_aug = torch.empty_like(X)
        # per-sample random segmentation + permutation
        for b in range(batches):
            n_segs = int(torch.randint(2, 5, (1,)).item())
            pts = np.linspace(0, timesteps, n_segs + 1, dtype=int)
            perm = np.random.permutation(n_segs)
            pieces = [X[b, pts[i]:pts[i+1], :] for i in perm]
            X_aug[b] = torch.cat(pieces, dim=0)
        return X_aug
    if augment_type == "cropping":
        keep = int(timesteps * (1 - scale_factor))
        start = int(torch.randint(0, timesteps - keep + 1, (1,)).item())
        cropped = X[:, start:start + keep, :]
        # pad or trim to keep shape (B, T, C)
        if cropped.shape[1] < timesteps:
            pad = torch.zeros(batches, timesteps - cropped.shape[1], cols, device=device)
            return torch.cat([cropped, pad], dim=1)
        else:
            return cropped
    if augment_type == "masking":
        mask = (torch.rand(batches, timesteps, cols, device=device) < scale_factor)
        X_aug = X.clone()
        X_aug[mask] = 0.0
        return X_aug
    if augment_type == "drift":
        drift = torch.linspace(0, float(scale_factor), timesteps, device=device).view(1, timesteps, 1)
        sign = 1.0 if torch.rand(1, device=device) < 0.5 else -1.0
        return X + sign * drift
    raise ValueError(f"Unknown augment type {augment_type}")

def compute_byol_loss(p_online: torch.Tensor, z_target: torch.Tensor) -> torch.Tensor:
    """Minimal BYOL loss: MSE between online predictions and target projections, adapted from the BYOL paper.
    Args:
        p_online: prediction from online network (batch, dim)
        z_target: projection from target network (batch, dim)
    Returns: Scalar loss"""
    # normalize for stability (optional but standard)
    p_online = F.normalize(p_online, dim=1)
    z_target = F.normalize(z_target, dim=1)

    z_target = z_target.detach() # stop gradients on target
    return 2 - 2 * (p_online * z_target).sum(dim=1).mean() # loss = MSE = 2 - 2 * cosine_sim

def update_target_encoding_ema(target_encoder: torch.nn.Module, online_encoder: torch.nn.Module, decay: float):
    """In-place EMA update of target params: target = decay*target + (1-decay)*online"""
    # return decay * target_encoding + (1 - decay) * new_values

    with torch.no_grad():
        for t_param, o_param in zip(target_encoder.parameters(), online_encoder.parameters()):
            t_param.data.mul_(decay).add_(o_param.data * (1.0 - decay))

def get_loss_weights(step, warmup_steps, max_steps): #can have schedule as linear OR cosine
    if step < warmup_steps:
        return dict(recon=1.0, contrast=0.0, pred=0.0)
    else:
        t = (step - warmup_steps) / (max_steps - warmup_steps)
        return dict(
            recon=max(0.1, 1.0 - t),   # decay recon to 0.1
            contrast=min(0.5, t),      # grow contrast to 0.5
            pred=min(1.0, t))           # grow pred to 1.0


In [ ]:
"[to remove] Run custom architecture with TS2Vec encoder"
# 0. params
w_pred, w_recon, w_contrast = 0.5, 0.2, 0.3
decoder_hidden_dims = [16, 64, 128]
optimizer_lr   = 9e-3
projection_dim = 16
proj_input_dim = projection_dim

# 1. augment X
X_1 = make_augmentations(torch.tensor(X_train, dtype=torch.float32, device=device), "jitter", device, 0.1)
X_2 = make_augmentations(torch.tensor(X_train, dtype=torch.float32, device=device), "mag_warp", device, 0.1)

# 2. fit TS2Vec on train data
ts2vec_encoder = TS2VecEncoder(z_pooling=z_pooling_method, device=device)
ts2vec_encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_output_dims,
                   depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=ts2vec_epochs)

# encode augmented views (X → z)
z_1 = torch.tensor(ts2vec_encoder.encode(X_1.cpu().numpy()), dtype=torch.float32, device=device)
z_2 = torch.tensor(ts2vec_encoder.encode(X_2.cpu().numpy()), dtype=torch.float32, device=device)

# encode full train/test set for supervised loss
z_train   = torch.tensor(ts2vec_encoder.encode(X_train), dtype=torch.float32, device=device)
z_test    = torch.tensor(ts2vec_encoder.encode(X_test), dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32, device=device)
y_test_t  = torch.tensor(y_test_scaled, dtype=torch.float32, device=device)

# 3. projection head
proj_head = ProjectionHead(input_dim=z_1.shape[1], proj_dim=projection_dim).to(device)
h_1 = proj_head(z_1)
h_2 = proj_head(z_2)

# 4. predictor for BYOL contrast
predictor = ProjectionHead(input_dim=proj_input_dim, proj_dim=projection_dim).to(device)
p_1 = predictor(h_1)

# 5. BYOL contrastive loss
loss_contrast = compute_byol_loss(p_1, h_2.detach())

# 6. optional decoder
decoder = Decoder(latent_dim=z_1.shape[1], output_shape=(X_train.shape[1], X_train.shape[2]),
                  hidden_sizes= decoder_hidden_dims).to(device)
x_recon = decoder(z_1)
loss_recon = F.mse_loss(x_recon, torch.tensor(X_train, dtype=torch.float32, device=device))

# 7. supervised predictor head
supervised_head = MLPHead(input_dim=z_1.shape[1], output_dim=y_train_scaled.shape[1],
                          hidden_sizes=predictor_hidden_sizes, lr=predictor_lr,
                          epochs=predictor_epochs, dropout=predictor_dropout, device=device)

# forward pass for supervised prediction
y_hat_train = supervised_head.model(z_train)
loss_pred   = F.mse_loss(y_hat_train, y_train_t)

# 8. combine losses
# total_loss = w_pred * loss_pred + w_recon * loss_recon + w_contrast * loss_contrast
weights = get_loss_weights(step, warmup_steps, max_steps)
loss    = (weights["recon"] * loss_recon + weights["contrast"] * loss_contrast + weights["pred"] * loss_pred)
print(f"Loss: pred={loss_pred.item():.4f}, recon={loss_recon.item():.4f}, contrast={loss_contrast.item():.4f}, total={loss.item():.4f}")

# 9. backprop and joint optimization
optimizer = torch.optim.AdamW(list(proj_head.parameters()) +
                              list(predictor.parameters()) +
                              list(decoder.parameters()) +
                              list(supervised_head.model.parameters()),
                              lr=optimizer_lr)
optimizer.zero_grad()
loss.backward()
optimizer.step()

"Predictions"
# 1️⃣ Encode test data with TS2Vec (frozen encoder)
z_test = torch.tensor(ts2vec_encoder.encode(X_test), dtype=torch.float32, device=device)

# 2️⃣ Predict with supervised MLPHead
y_pred = supervised_head.predict(z_test) # changes done to supervised_head

rmse  = root_mean_squared_error(y_test_scaled, y_pred)
print(f"Custom architecture RMSE: {rmse:.4f}")


Epoch #0: loss=3.756615025656564
Epoch #1: loss=3.7478088991982594
Epoch #2: loss=3.767908368791853
Epoch #3: loss=3.7140105111258372
Epoch #4: loss=3.73184905733381
Epoch #5: loss=3.5182552337646484
Epoch #6: loss=3.7044179439544678
Epoch #7: loss=3.5453496319907054
Epoch #8: loss=3.526351043156215
Epoch #9: loss=3.5637972014290944
Epoch #10: loss=3.6841799191066196
Epoch #11: loss=3.5505950450897217
Epoch #12: loss=3.493210417883737
Epoch #13: loss=3.5747156143188477
Epoch #14: loss=3.5476204667772566
Epoch #15: loss=3.326601948056902
Epoch #16: loss=3.4455318110329762
Epoch #17: loss=3.443331718444824
Epoch #18: loss=3.4230355194636752
Epoch #19: loss=3.455465691430228
Loss: pred=0.9780, recon=0.9996, contrast=2.7649, total=1.5184


In [ ]:
"""Loop custom architecture"""
# params
train_epochs = 50
batch_size  = 16
warmup_steps= 1000
max_steps   = train_epochs * (len(X_train) // batch_size)
step        = 0

decoder_hidden_dims = [16, 64, 128]
optimizer_lr   = 9e-3
projection_dim = 16

# init methods
encoder = TS2VecEncoder(z_pooling=z_pooling_method, device=device)
encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_output_dims,
            depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=ts2vec_epochs)

proj_head   = ProjectionHead(input_dim=ts2vec_output_dims, proj_dim=projection_dim).to(device)
predictor   = ProjectionHead(input_dim=projection_dim, proj_dim=projection_dim).to(device)
decoder     = Decoder(latent_dim=ts2vec_output_dims, output_shape=(X_train.shape[1], X_train.shape[2]),
                      hidden_sizes=decoder_hidden_dims).to(device)
supervised_head = MLPHead(input_dim=ts2vec_output_dims, output_dim=y_train_scaled.shape[1], hidden_sizes=predictor_hidden_sizes,
                          lr=predictor_lr, epochs=1, dropout=predictor_dropout, device=device)#.to(device)
# joint optimizer
params = (list(proj_head.parameters()) +
          list(predictor.parameters()) +
          list(decoder.parameters()) +
          list(supervised_head.model.parameters()))
optimizer = torch.optim.AdamW(params, lr=optimizer_lr)

for epoch in range(train_epochs):
    for i in range(0, len(X_train), batch_size):
        step   += 1
        X_batch = X_train[i:i+batch_size]
        y_batch = y_train_scaled[i:i+batch_size]

        # 1. Augment
        X1 = make_augmentations(torch.tensor(X_batch, dtype=torch.float32, device=device), "jitter", device, 0.1)
        X2 = make_augmentations(torch.tensor(X_batch, dtype=torch.float32, device=device), "mag_warp", device, 0.1)

        # 2. Encode
        z1 = torch.tensor(encoder.encode(X1.cpu().numpy()), dtype=torch.float32, device=device)
        z2 = torch.tensor(encoder.encode(X2.cpu().numpy()), dtype=torch.float32, device=device)

        # 3. Projections
        h1, h2 = proj_head(z1), proj_head(z2)
        p1     = predictor(h1)

        # 4. Losses
        loss_contrast = compute_byol_loss(p1, h2.detach())
        x_recon       = decoder(z1)
        loss_recon    = F.mse_loss(x_recon, torch.tensor(X_batch, dtype=torch.float32, device=device))

        y_batch_t = torch.tensor(y_batch, dtype=torch.float32, device=device)
        y_hat     = supervised_head.model(z1)
        loss_pred = F.mse_loss(y_hat, y_batch_t)

        # 5. Weighted total loss
        weights = get_loss_weights(step, warmup_steps, max_steps)
        loss    = (weights["recon"] * loss_recon + weights["contrast"] * loss_contrast + weights["pred"] * loss_pred)

        # 6. Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{train_epochs}: "
          f"pred={loss_pred.item():.4f}, recon={loss_recon.item():.4f}, "
          f"contrast={loss_contrast.item():.4f}, total loss={loss.item():.4f}")

# ===== Inference =====
proj_head.eval()
predictor.eval()
decoder.eval()
supervised_head.model.eval()

with torch.no_grad():
    # encode test set once with frozen encoder
    z_test = torch.tensor(ts2vec_encoder.encode(X_test), dtype=torch.float32, device=device)
    y_pred = supervised_head.model(z_test).cpu().numpy()

rmse = root_mean_squared_error(y_test_scaled, y_pred)
print(f"Custom architecture RMSE: {rmse:.4f}")



Epoch #0: loss=3.743434429168701
Epoch #1: loss=3.613430908748082
Epoch #2: loss=3.5072527953556607
Epoch #3: loss=3.4945831980024065
Epoch #4: loss=3.567568676812308
Epoch #5: loss=3.686833688191005
Epoch #6: loss=3.4752931935446605
Epoch #7: loss=3.52622846194676
Epoch #8: loss=3.2466952800750732
Epoch #9: loss=3.5197178295680454
Epoch #10: loss=3.5428692272731235
Epoch #11: loss=3.3559439522879466
Epoch #12: loss=3.4398673261914934
Epoch #13: loss=3.4161771706172397
Epoch #14: loss=3.417814459119524
Epoch #15: loss=3.3192132200513567
Epoch #16: loss=3.3812051500592912
Epoch #17: loss=3.1351165090288435
Epoch #18: loss=3.3130920273917064
Epoch #19: loss=3.42576721736363
Epoch 1/50: pred=0.8478, recon=0.9419, contrast=2.2015, total loss=0.9419
Epoch 2/50: pred=0.8484, recon=0.9267, contrast=2.1975, total loss=0.9267
Epoch 3/50: pred=0.8475, recon=0.8995, contrast=2.1993, total loss=0.8995
Epoch 4/50: pred=0.8486, recon=0.9034, contrast=2.1980, total loss=0.9034
Epoch 5/50: pred=0.8481

In [ ]:
# running the custom architecture
z_pooling_method   = "mean"
ts2vec_hidden_dims = 16 # units in each layer (> than latent dim)
ts2vec_output_dims = 8 # latent dim
ts2vec_depth       = 2 # num layers
ts2vec_batch_size  = 32
ts2vec_epochs      = 20

predictor_lr       = 0.009
predictor_epochs   = 150
predictor_dropout  = 0.05
predictor_hidden_sizes = [32, 48, 64] # latent to y output

# 1. Fit TS2Vec
from other_encoders.ts2vec_encoder import TS2VecEncoder

ts2vec_encoder = TS2VecEncoder(z_pooling=z_pooling_method, device=device)
ts2vec_encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_output_dims,
                   depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=ts2vec_epochs)
z_train = ts2vec_encoder.encode(X_train)
z_test  = ts2vec_encoder.encode(X_test)

# Train NN predictor
nn_predictor = MLPHead(input_dim=z_train.shape[1], output_dim=y_train_scaled.shape[1],
                           hidden_sizes=predictor_hidden_sizes, lr=predictor_lr,
                           epochs=predictor_epochs, dropout=predictor_dropout, device=device)
nn_predictor.train(z_train, y_train_scaled, z_test, y_test_scaled)
nn_rmse = nn_predictor.evaluate(z_test, y_test_scaled)


# ====== plain predictors ======
linreg_loss, catboost_loss, unsupervised_rmse, rf_rmse, el_rmse = Preds.evaluate_models_on_dataset(z_train, y_train_scaled,
                                                                                                   z_test, y_test_scaled)
print(f"dataset: {desired_dataset}, method: ts2vec")
print( "    RMSE   | LinReg | CatBoost | Cluster | RForest | ElasticNet | NN")
print(f"& Z (ts2vec) & {linreg_loss:.4f} & {catboost_loss:.4f} & {unsupervised_rmse:.4f}  & {rf_rmse:.4f}  & {el_rmse:.4f} \
    & {nn_rmse:.4f} \\ ")


In [ ]:
"""[real data] Conditional VAE. Train on train set, inference on test set"""

should_we_include_X = True

# === Data parameters ===
batch_size = 32

# === Model parameters ===
hidden_dim = 64
latent_dim = 10
dropout    = 0.05

# === Training parameters ===
epochs              = 100
learning_rate       = 1e-3
weight_decay        = 1e-5
scheduler_patience  = 5
early_stop_patience = 20
save_path           = "best_cvae.pth"

# === Dataset from df (downsample + split) ===
df_small = df.sample(frac=0.1, random_state=42)  # keep 10%
X = df_small.drop(columns=y_cols + [time_col_name], errors="ignore").values
y = df_small[y_cols].values

# === dont use traintestsplit for timeseries (it randomly shuffles, breaking temporal order)
split_idx       = int(len(df_small) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# === Scaling ===
x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_train  = x_scaler.fit_transform(X_train)
X_test   = x_scaler.transform(X_test)
y_train  = y_scaler.fit_transform(y_train)
y_test   = y_scaler.transform(y_test)

# === Convert to tensors ===
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)
y_dim   = y_train.shape[1]
x_dim   = X_train.shape[1]

# === Datasets + Loaders ===
train_dataset = TensorDataset(y_train, X_train)
test_dataset  = TensorDataset(y_test, X_test)
train_loader  = DataLoader(train_dataset, batch_size=batch_size, drop_last=True)
val_loader    = DataLoader(test_dataset,  batch_size=batch_size, drop_last=True)

# === Model + Optimizer + Scheduler===
cond_vae  = ae.ConditionalVAE(y_dim=y_dim, x_dim=x_dim, hidden_dim=hidden_dim, latent_dim=latent_dim, dropout=dropout).to(device)
optimizer = torch.optim.AdamW(cond_vae.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=scheduler_patience)

# === Trainer ===
trainer   = train_ae.TrainConditionalVAE()
best_loss = trainer.train_cvae(device=device, cvae=cond_vae, epochs=epochs,train_loader=train_loader, optimizer=optimizer,
                               scheduler=scheduler, val_loader=val_loader, patience=early_stop_patience, save_path=save_path)
print(f"Best validation loss: {best_loss:.4f}")

# === inference ===
cond_vae.load_state_dict(torch.load(save_path))
cond_vae.eval()

# === Full test set prediction ===
batch_size_pred   = 64  # adjust based on memory
all_y_pred_scaled = []

with torch.no_grad():
    for batch in range(0, len(X_test), batch_size_pred):
        x_batch = X_test[batch : batch + batch_size_pred].to(device)
        if not should_we_include_X:
            x_batch = torch.zeros(x_batch.size(0), x_dim).to(device)
        z_batch             = torch.randn(x_batch.size(0), latent_dim).to(device)
        y_batch_pred_scaled = cond_vae.decode(z_batch, x=x_batch)
        all_y_pred_scaled.append(y_batch_pred_scaled.cpu())

# Concatenate all batches + MSE
y_pred_scaled = torch.cat(all_y_pred_scaled, dim=0).numpy()
mse = mean_squared_error(y_test, y_pred_scaled)
mae = mean_absolute_error(y_test, y_pred_scaled)
print(f"Scaled y: test MSE = {mse:.4f}, test MAE = {mae:.4f}")

y_pred      = y_scaler.inverse_transform(y_pred_scaled)
y_true_orig = y_scaler.inverse_transform(y_test.numpy())

plt.plot(y_true_orig, label="True")
plt.plot(y_pred, label="Predicted")
plt.title(f"Cond. VAE ('{desired_dataset}' dataset)")
plt.xlabel("Timestep")
plt.ylabel("y value")
plt.legend()
plt.show()


# NOTE: consider this repo for Conditional VAE (https://github.com/unnir/cVAE/blob/master/cvae.py)
# or https://freedium.cfd/https://medium.com/@sofeikov/implementing-conditional-variational-auto-encoders-cvae-from-scratch-29fcbb8cb08f

In [ ]:
"""Multistep LSTM predictor"""
import torch.optim as optim
from forecasting_module import BaseForecaster

class FlexibleLSTM(nn.Module):
    def __init__(self, input_dim: int, hidden_size: int, output_dim: int):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])  # last step only

class FlexibleLSTMTrainer:
    def __init__(self, model, lr=1e-3, epochs=30, device="cpu"):
        self.model = model.to(device)
        self.lr = lr
        self.epochs = epochs
        self.device = device
        self.loss_fn = nn.MSELoss()

    def _make_batches(self, y, X, seq_len, horizon):
        """Convert timeseries to (X_seq, y_future) windows."""
        data = y if X is None else np.concatenate([y, X], axis=1)
        X_batches, y_batches = [], []
        for i in range(len(data) - seq_len - horizon):
            X_batches.append(data[i:i+seq_len])
            y_batches.append(y[i+seq_len:i+seq_len+horizon])
        return (torch.tensor(np.stack(X_batches), dtype=torch.float32),
                torch.tensor(np.stack(y_batches), dtype=torch.float32),)

    def fit(self, y_train, X_train, seq_len, horizon, y_val=None, X_val=None):
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        Xb, yb = self._make_batches(y_train, X_train, seq_len, horizon)
        Xb, yb = Xb.to(self.device), yb.to(self.device)

        for epoch in range(self.epochs):
            self.model.train()
            optimizer.zero_grad()
            preds = self.model(Xb)
            loss  = self.loss_fn(preds, yb[:, -1, :])  # predict horizon last-step
            loss.backward()
            optimizer.step()
            if (epoch+1) % 10 == 0:
                print(f"Epoch {epoch+1}, loss={loss.item():.4f}")

    def predict_seq2seq(self, y_test, X_test, seq_len, horizon):
        Xb, yb = self._make_batches(y_test, X_test, seq_len, horizon)
        self.model.eval()
        with torch.no_grad():
            preds = self.model(Xb.to(self.device)).cpu().numpy()
        return preds

    def predict_autoreg(self, y_test, X_test, seq_len, horizon):
        """Step-by-step forecasting with feedback of predictions."""
        data  = y_test if X_test is None else np.concatenate([y_test, X_test], axis=1)
        preds = []
        self.model.eval()
        with torch.no_grad():
            for i in range(len(data) - seq_len - horizon):
                window = torch.tensor(data[i:i+seq_len], dtype=torch.float32).unsqueeze(0).to(self.device)
                pred   = self.model(window).cpu().numpy()
                preds.append(pred)
                # feed prediction back only into y part (not exogenous)
                data[i+seq_len] = np.concatenate([pred[0], data[i+seq_len, y_test.shape[1]:]])
        return np.array(preds)


class LSTMForecaster(BaseForecaster):
    def __init__(self, df, time_col: str, y_cols: list[str],
                 use_exogenous: bool = False, seq_len: int = 30, horizon: int = 10,
                 hidden_size: int = 64, epochs: int = 30, lr: float = 1e-3):
        super().__init__(df, time_col, y_cols)
        self.use_exogenous = use_exogenous
        self.seq_len = seq_len
        self.horizon = horizon
        self.hidden_size = hidden_size
        self.epochs = epochs
        self.lr = lr

        # Exogenous columns = all columns minus time + y
        if self.use_exogenous:
            self.X_cols = [c for c in df.columns if c not in [time_col] + y_cols]
        else:
            self.X_cols = []

    def _prepare_data(self, df):
        """Return tensors for y (and X if exogenous)."""
        y = df[self.y_cols].values.astype(np.float32)
        if self.use_exogenous and self.X_cols:
            X = df[self.X_cols].values.astype(np.float32)
        else:
            X = None
        return y, X

    def fit(self, df_train, df_val=None):
        y_train, X_train = self._prepare_data(df_train)
        y_val, X_val = (None, None) if df_val is None else self._prepare_data(df_val)

        self.model = FlexibleLSTM(
            input_dim=len(self.y_cols) + (X_train.shape[1] if X_train is not None else 0),
            hidden_size=self.hidden_size,
            output_dim=len(self.y_cols))

        trainer = FlexibleLSTMTrainer(self.model, lr=self.lr, epochs=self.epochs)
        trainer.fit(y_train, X_train, self.seq_len, self.horizon,
                    y_val=y_val, X_val=X_val)
        self.trainer = trainer

    def predict(self, df_test, autoregressive: bool = False):
        y_test, X_test = self._prepare_data(df_test)
        if autoregressive:
            return self.trainer.predict_autoreg(y_test, X_test, self.seq_len, self.horizon)
        else:
            return self.trainer.predict_seq2seq(y_test, X_test, self.seq_len, self.horizon)

    def forecast_lstm(self, train_df, test_df, horizon: int,
                      use_exogenous: bool = True, stepwise: bool = False):
        """Train + forecast like SARIMAX.
        Args:
            train_df, test_df: pandas DataFrames
            horizon: forecast horizon
            use_exogenous: toggle exogenous features
            stepwise: if True → autoregressive, else → direct multi-step
        Returns:
            forecast_dict: {y_col: np.array predictions}
            y_true: true target values (scaled)"""
        # override exogenous toggle for this run
        self.use_exogenous = use_exogenous
        if use_exogenous:
            self.X_cols = [c for c in train_df.columns if c not in [self.time_col] + self.y_cols]
        else:
            self.X_cols = []

        self.fit(train_df)
        preds = self.predict(test_df, autoregressive=stepwise)
        y_true, _ = self._prepare_data(test_df)

        if preds.ndim == 1:
            preds = preds.reshape(-1, 1)

        forecast_dict = {col: preds[:, i] for i, col in enumerate(self.y_cols)}
        self.forecast_dfs = {col: forecast_dict[col] for col in self.y_cols}
        return forecast_dict, y_true


"""LSTM run"""
n_windows = 5
horizon = 10       # prediction steps
seq_len = 30       # past steps
# min_window_len = seq_len + horizon  # ensure each window is long enough

logging.basicConfig(level=logging.INFO)

forecaster = LSTMForecaster(df_uniform, time_col=time_col_name, y_cols=y_cols, seq_len=seq_len)

print("==== single window evaluation ====")
# forecast_dict, y_true_scaled = forecaster.forecast_lstm(
#     df_train, df_test, horizon,
#     use_exogenous=True,   # or False
#     stepwise=False)        # sequence prediction (False) or step-by-step (True)

# forecast_df_temp = forecaster.forecast_dfs[y_cols[0]]

print("==== multi window evaluation ====")
all_forecasts, mae_list, sizes = {}, [], []
windows_list = ForecastUtils.make_windows(
    df_uniform, n_windows=n_windows, horizon_len=horizon,
    horizon_frac=horizon_frac, min_window_len=min_window_len, start_point=0)

for i, (train_df, test_df) in enumerate(windows_list):
    print(f"Window {i}: train {train_df.shape}, test {test_df.shape}")
    forecast_dict, y_true_scaled = forecaster.forecast_lstm(
        train_df, test_df, horizon,
        use_exogenous=True,
        stepwise=True)
    all_forecasts[f"window_{i}"] = forecast_dict

    for j, target in enumerate(y_cols):
        y_true = y_true_scaled[:, j]
        y_pred = forecast_dict[target]
        mae = mean_absolute_error(y_true, y_pred)
        mae_list.append(mae)
        sizes.append(len(y_true))

# ---- Weighted MAE ----
weighted_mae = sum(MAE * n for MAE, n in zip(mae_list, sizes)) / sum(sizes)
std_mae      = math.sqrt(sum((MAE - weighted_mae) ** 2 * n for MAE, n in zip(mae_list, sizes)) / sum(sizes))
print(f"Final MAE: mean ± std= {{{weighted_mae:.4f}}}{{{std_mae:.4f}}}")


In [ ]:
"""TimesFM"""
from timesfm import TimesFmHparams, TimesFm, TimesFmCheckpoint

# Hyperparameters
hparams = TimesFmHparams(
    backend="jax",
    per_core_batch_size=32,
    horizon_len=128,
    num_layers=20,
    context_len=512,
    use_positional_embedding=True,)

# Local checkpoint folder containing 'checkpoint'
checkpoint = TimesFmCheckpoint(local_dir="interim_data")

# Initialize model
model = TimesFm(hparams=hparams, checkpoint=checkpoint)

# Load manually (if needed)
# model.load_from_checkpoint("interim_data/checkpoint", checkpoint_type=CheckpointType.FLAX)
model.load_from_checkpoint(repo_id="google/timesfm-1.0-200m")#, checkpoint_type=CheckpointType.FLAX)

# Forecast example
y = np.arange(100)
forecast = model.forecast(y, horizon=10)
print(forecast)


In [ ]:
"""Conditional VAE. Train on train set, inference on test set"""

# === Data parameters ===
n_samples  = 1000
y_dim      = 1
x_dim      = 10  # optional
batch_size = 32

# === Model parameters ===
hidden_dim = 64
latent_dim = 10
dropout    = 0.05

# === Training parameters ===
epochs              = 100
learning_rate       = 1e-3
weight_decay        = 1e-5
scheduler_patience  = 5
early_stop_patience = 10
save_path           = "best_cvae.pth"

# === Dataset ===
y_data  = torch.randn(n_samples, y_dim)
x_data  = torch.randn(n_samples, x_dim)
dataset = TensorDataset(y_data, x_data)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(dataset, batch_size=batch_size)

cond_vae  = ae.ConditionalVAE(y_dim=y_dim, x_dim=x_dim, hidden_dim=hidden_dim,
                              latent_dim=latent_dim, dropout=dropout).to(device)

# === Optimizer + Scheduler ===
optimizer = torch.optim.AdamW(cond_vae.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=scheduler_patience)

# === Trainer ===
trainer   = train_ae.TrainConditionalVAE()
best_loss = trainer.train_cvae(device=device, cvae=cond_vae, epochs=epochs, train_loader=train_loader, optimizer=optimizer,
                               scheduler=scheduler, val_loader=val_loader, patience=early_stop_patience, save_path=save_path)
print(f"Best validation loss: {best_loss:.4f}")

# === Load best model for inference ===
cond_vae.load_state_dict(torch.load(save_path))
cond_vae.eval()

# === Example inference ===
n_rows_gen   = 5
z_sample     = torch.randn(n_rows_gen, latent_dim).to(device)
is_x_present = True
if is_x_present:
    x_sample = torch.randn(n_rows_gen, x_dim).to(device)
else:
    x_sample = torch.zeros(n_rows_gen, x_dim).to(device)

y_sample = cond_vae.decode(z_sample, x=x_sample)
print(f"Generated y sample: {y_sample}")


In [ ]:
"""Applying metrics on X vs z"""

# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# # Split your original data
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Encode train/test via VAE
# z_train = vae.encoder.predict(X_train)  # shape (num_samples, latent_dim)
# z_test  = vae.encoder.predict(X_test)

# # --- Predictor on raw X ---
# model_X = LinearRegression()
# model_X.fit(X_train.reshape(X_train.shape[0], -1), y_train)  # flatten if needed
# y_pred_X = model_X.predict(X_test.reshape(X_test.shape[0], -1))

# # --- Predictor on latent z ---
# model_z = LinearRegression()
# model_z.fit(z_train, y_train)
# y_pred_z = model_z.predict(z_test)

# # --- Metrics ---
# def print_metrics(y_true, y_pred, name):
#     mse = mean_squared_error(y_true, y_pred)
#     mae = mean_absolute_error(y_true, y_pred)
#     r2  = r2_score(y_true, y_pred)
#     print(f"{name} -> MSE: {mse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")

# print_metrics(y_test, y_pred_X, "Predictor on X")
# print_metrics(y_test, y_pred_z, "Predictor on z")


In [ ]:
from downsampling import LTTBDownsampler, OtherDownsamplers

def df_plotter(df, df_downsampled, col_to_plot: int, time_col: str) -> None:
    plt.figure(figsize=(7, 4))
    plt.plot(df[time_col], df.iloc[:, col_to_plot], alpha=0.8, linewidth=2)
    plt.plot(df_downsampled[time_col], df_downsampled.iloc[:, col_to_plot], alpha=0.8, linewidth=1)
    plt.legend(['Original', 'Downsampled'])
    plt.tight_layout()
    plt.show()

def _add_time_features(df: pd.DataFrame, y_col: str, time_col: str = 'Datetime', lag_amount: int = 1, rolling_window: int = 3) -> pd.DataFrame:
    df = df.copy()
    df.loc[:, 'hour']      = df[time_col].dt.hour
    df.loc[:, 'dayofweek'] = df[time_col].dt.dayofweek
    df.loc[:, f'lag_{lag_amount}'] = df[y_col].shift(lag_amount)
    df.loc[:, f'rolling_mean_{rolling_window}'] = df[y_col].rolling(rolling_window).mean()
    return df.dropna()


class SplitterScaler:
    @staticmethod
    def split_X_and_y(df, time_col: str, y) -> tuple:
        X = df.drop([time_col]+ y, axis=1)
        y = df[y]
        return X, y
        # y1= df[['PowerConsumption_Zone1']]
        # y2= df[['PowerConsumption_Zone2']]
        # y3= df[['PowerConsumption_Zone3']]
        # return X, y, y1, y2, y3

    @staticmethod
    def traintest_split_then_scale(X: pd.DataFrame, y: pd.Series) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, StandardScaler]:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        scaler_X       = StandardScaler()
        X_train_scaled = scaler_X.fit_transform(X_train)
        X_test_scaled  = scaler_X.transform(X_test)

        scaler_y       = StandardScaler()
        y_train_scaled = scaler_y.fit_transform(y_train)
        y_test_scaled  = scaler_y.transform(y_test)
        return X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled, scaler_y

    @staticmethod
    def scale_X_and_y(X: pd.DataFrame, y: pd.DataFrame):
        """Scale features X and target y using StandardScaler.
        Returns scaled arrays and fitted scalers"""
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        X_scaled = scaler_X.fit_transform(X)
        y_scaled = scaler_y.fit_transform(y)
        return X_scaled, y_scaled, (scaler_X, scaler_y)

    @staticmethod
    def scale_X_and_y_latents(X_train: pd.DataFrame | np.ndarray, X_valid: pd.DataFrame | np.ndarray,
                            y_train: pd.DataFrame | np.ndarray, y_valid: pd.DataFrame | np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, StandardScaler, StandardScaler]:
        """Fit scalers on X_train and y_train, transform both train and valid sets.
        Returns scaled X_train, X_valid, y_train, y_valid, scaler_X, scaler_y"""
        scaler_X  = StandardScaler().fit(X_train)
        X_train_s = scaler_X.transform(X_train)
        X_valid_s = scaler_X.transform(X_valid)

        y_train   = np.asarray(y_train).reshape(-1, 1)
        y_valid   = np.asarray(y_valid).reshape(-1, 1)
        scaler_y  = StandardScaler().fit(y_train)
        y_train_s = scaler_y.transform(y_train)
        y_valid_s = scaler_y.transform(y_valid)

        return X_train_s, X_valid_s, y_train_s, y_valid_s, scaler_X, scaler_y


class TrainerAndEvaluator:

    @staticmethod
    def train_catboost_model(X_train_scaled, y_train_scaled):
        model = CatBoostRegressor(iterations=1300, depth=8, learning_rate=0.15, l2_leaf_reg=2, loss_function='RMSE',
                                  random_seed=42, verbose=0, early_stopping_rounds=100, task_type='CPU', bagging_temperature=3)
        model.fit(X_train_scaled, y_train_scaled.ravel())
        return model

    @staticmethod
    def train_and_evaluate_model(train_df: Optional[pd.DataFrame] = None, test_df: Optional[pd.DataFrame] = None,
                                 X_train: Optional[np.ndarray] = None, X_test: Optional[np.ndarray] = None,
                                 y_train: Optional[np.ndarray] = None, y_test: Optional[np.ndarray] = None,
                                 scaler_X: Optional[StandardScaler] = None) -> Tuple[float, float]:
        """Train and evaluate CatBoost. If X/y provided, use them directly.
        Otherwise, extract features from train_df/test_df"""
        
        if X_train is None or y_train is None:
            # Mode 1: from DataFrames
            X, y = SplitterScaler.split_X_and_y(train_df, time_col_name, y_cols)
            X_train_s, X_test_s, y_train_s, y_test_s, _ = SplitterScaler.traintest_split_then_scale(X, y)
        else:
            # Mode 2: from arrays
            if scaler_X:
                X_train_s = scaler_X.transform(X_train)
                X_test_s  = scaler_X.transform(X_test)
            else:
                X_train_s, X_test_s = X_train, X_test  # already scaled
            y_train_s, y_test_s = y_train, y_test

        model  = TrainerAndEvaluator.train_catboost_model(X_train_s, y_train_s)
        y_pred = model.predict(X_test_s)

        # --- metrics ---
        if y_test_s.ndim == 1 or y_test_s.shape[1] == 1:
            rmse  = float(np.sqrt(mean_squared_error(y_test_s, y_pred)))
            nrmse = float(rmse / (y_test_s.max() - y_test_s.min()))
        else:
            rmses = [np.sqrt(mean_squared_error(y_test_s[:, i], y_pred[:, i]))
                     for i in range(y_test_s.shape[1])]
            rmse  = float(np.mean(rmses))
            nrmse = float(np.mean([
                r / (y_test_s[:, i].max() - y_test_s[:, i].min())
                for i, r in enumerate(rmses)]))
        return rmse, nrmse

def downsample_and_evaluate(df, new_points_per_col, compression_ratio_list, rmse_list, nrmse_list, method, rdp_epsilon=0.1):
    """Downsample df, evaluate model on original and compressed, store metrics."""
    if method == "union_lttb":
        df_downsampled = LTTBDownsampler.downsample_using_lttb_union(df, time_col=time_col_name, max_points_per_col=new_points_per_col)
    elif method == "intersection_lttb":
        df_downsampled = LTTBDownsampler.downsample_using_lttb_intersection(df, time_col=time_col_name, max_points_per_col=new_points_per_col)
    elif method == "union_lttb_with_spikes":
        df_downsampled = LTTBDownsampler.downsample_lttb_union_with_spikes(df, time_col=time_col_name, max_points_per_col=new_points_per_col,\
                                                                           spike_thresh=4, spike_method='mad')
    elif method == "union_rdp":
        df_downsampled = OtherDownsamplers.downsample_using_rdp_union(df, time_col=time_col_name, epsilon=rdp_epsilon)
    else:
        raise ValueError(f"Unknown downsampling method: {method}")

    rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(df)
    # print(f"orig. RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

    compression_ratio = len(df) / len(df_downsampled)
    # print(f"data compressed {compression_ratio:.2f}x")
    rmse_c, nrmse_c = TrainerAndEvaluator.train_and_evaluate_model(df_downsampled)
    # print(f"compress. RMSE: {rmse_c:.3f}, norm RMSE: {nrmse_c:.3%}")

    compression_ratio_list.append(compression_ratio)
    rmse_list.append(rmse_c)
    nrmse_list.append(nrmse_c)
    return compression_ratio_list, rmse_list, nrmse_list


In [ ]:
"""timeVAE preprocessing (windowing)"""

def create_windows(data: np.ndarray, window_size: int, step_size: int) -> np.ndarray:
    N      = (data.shape[0] - window_size) // step_size + 1
    windows= np.array([data[i*step_size:i*step_size+window_size] for i in range(N)])
    return windows

X_raw    = df.drop(columns=[time_col_name] + y_cols).values
y_raw    = df[y_cols].values

# scale + Transform
scaler_X = StandardScaler().fit(X_raw)
scaler_y = StandardScaler().fit(y_raw)
X_scaled = scaler_X.transform(X_raw)
y_scaled = scaler_y.transform(y_raw)

window_size = 20
step_size   = window_size // 4
X_windows   = create_windows(X_scaled, window_size, step_size)
y_windows   = create_windows(y_scaled, window_size, step_size)

print("X_win shape:", X_windows.shape, "y_win shape:", y_windows.shape)

# Save to ./data/ + timeVAE repo
should_we_save_in_timeVAE = True
if should_we_save_in_timeVAE:
    os.makedirs('../interim_data', exist_ok=True)
    np.savez_compressed(f"../interim_data/{desired_dataset}.npz", data=np.array(X_windows, dtype=np.float32))
    # np.savez_compressed(f"../interim_data/{desired_dataset}.npz", data=X_windows, target=y_windows)
    np.savez_compressed(f"../../timeVAE/data/{desired_dataset}.npz", data=np.array(X_windows, dtype=np.float32))
print(f"saved {desired_dataset} info to ../interim_data/{desired_dataset}.npz")


In [ ]:
"""predicting y on latents z"""
z_train = np.load(f"../interim_data/z_train_{desired_dataset}.npy")
z_valid = np.load(f"../interim_data/z_valid_{desired_dataset}.npy")
print(f"z_train shape: {z_train.shape}, z_valid shape: {z_valid.shape}")

y_squeezed = y_windows[:, -1, 0].reshape(-1, 1)

# y train/valid shaped like z_train/valid
n_train = z_train.shape[0]
n_valid = z_valid.shape[0]
y_train = y_squeezed[:n_train]
y_valid = y_squeezed[n_train:n_train + n_valid]

print("y_train/y_valid shapes:", y_train.shape, y_valid.shape)

# z_train_df = pd.DataFrame(z_train, columns=[f"z{i}" for i in range(z_train.shape[1])])
# z_valid_df = pd.DataFrame(z_valid, columns=[f"z{i}" for i in range(z_valid.shape[1])])

rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(X_train=z_train, X_test=z_valid,
                                                           y_train=y_train, y_test=y_valid)
print(f"RMSE: {rmse:.4f}, NRMSE: {nrmse:.4f}")


In [ ]:
"""catch22"""

def catch22_features_from_windows(X_windows: np.ndarray, y_windows: np.ndarray, which_y: str) -> tuple[np.ndarray, np.ndarray]:
    """Apply catch22 to each window/channel and return (features, targets).
    X_windows: shape (n_windows, window_size, n_channels)
    y_windows: shape (n_windows, window_size, n_targets)"""
    
    n_windows, _, n_channels = X_windows.shape
    feats = np.empty((n_windows, n_channels * 22), dtype=float)
    
    for i in range(n_windows):
        channel_feats = [catch22_all(X_windows[i, :, ch])['values']
                         for ch in range(n_channels)]
        feats[i] = np.concatenate(channel_feats)
    
    # Take last value in each target window
    if which_y == "last":
        y_out = y_windows[:, -1, :]
    elif which_y == "mean":
        y_out = np.mean(y_windows, axis=1)
    return feats, y_out

X_c22, y_c22 = catch22_features_from_windows(X_windows, y_windows, which_y="last")
X_train, X_test, y_train, y_test = train_test_split(X_c22, y_c22, test_size=0.2, random_state=42)
print(f"shapes: X_train {X_train.shape}, X_test {X_test.shape}, y_train {y_train.shape}, y_test {y_test.shape}")

# X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled, _ = SplitterScaler.traintest_split_then_scale(X_c22, y_c22)

if y_train.shape[1] == 1:
    rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(X_train=X_train, X_test=X_test,
                                                               y_train=y_train.ravel(), y_test=y_test.ravel())
    print(f"Train RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")
else:
    rmses   = []
    nrmse_s = []
    for i in range(y_train.shape[1]):
        rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(
            X_train=X_train,
            X_test=X_test,
            y_train=y_train[:, i],
            y_test=y_test[:, i])
        rmses.append(rmse)
        nrmse_s.append(nrmse)
    print("Mean RMSE:", np.mean(rmses))
    print("Mean NRMSE:", np.mean(nrmse_s))


In [ ]:
"""(outdated) Evaluate downsampling methods"""
df_train, df_test = train_test_split(df, test_size=0.2, shuffle=False)
rmse, nrmse       = TrainerAndEvaluator.train_and_evaluate_model(df_train, df_test)
print(f"Original train RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

N_VW   = 10

N_LTTB = 2
spike_method      = 'zscore' # 'zscore' or 'mad'
spike_method_thres= 3

rdp_epsilon       = 1
subsampling_ratio = 100
N_last_rows       = 100

lttb_union_train = LTTBDownsampler.downsample_using_lttb_union(df_train, time_col=time_col_name,
                                                               max_points_per_col=len(df_train) // N_LTTB)
rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(lttb_union_train, df_test)
print(f"LTTB union ({N_LTTB=})")
print(f"data compressed {len(df_train)/len(lttb_union_train):.2f}x")
print(f"compress. RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

lttb_intersect_train = LTTBDownsampler.downsample_using_lttb_intersection(df_train, time_col=time_col_name,
                                                                          max_points_per_col=len(df_train) // N_LTTB)
rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(lttb_intersect_train, df_test)
print("LTTB intersection")
print(f"data compressed {len(df_train)/len(lttb_intersect_train):.2f}x")
print(f"compress. RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

lttb_union_spikes_train = LTTBDownsampler.downsample_lttb_union_with_spikes(df_train, time_col=time_col_name,
                                                                            max_points_per_col=len(df_train) // N_LTTB,
                                                                            spike_thresh=spike_method_thres, spike_method=spike_method)
rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(lttb_union_spikes_train, df_test)
print(f"LTTB union +spikes ({spike_method=})")
print(f"data compressed {len(df_train)/len(lttb_union_spikes_train):.2f}x")
print(f"compress. RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

rdp_union_train = OtherDownsamplers.downsample_using_rdp_union(df_train, time_col=time_col_name, epsilon=rdp_epsilon)
rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(rdp_union_train, df_test)
print(f"RDP union ({rdp_epsilon=})")
print(f"data compressed {len(df_train)/len(rdp_union_train):.2f}x")
print(f"compress. RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

vw_union_train = OtherDownsamplers.downsample_using_vw_union(df_train, time_col=time_col_name, target_points=len(df_train) // N_VW)
rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(vw_union_train, df_test)
print(f"VW union ({N_VW=})")
print(f"data compressed {len(df_train)/len(vw_union_train):.2f}x")
print(f"compress. RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

subsample_train = OtherDownsamplers.subsample_timeseries(df_train, subsampling_ratio=subsampling_ratio)
rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(subsample_train, df_test)
print(f"Subsample ({subsampling_ratio=})")
print(f"data compressed {len(df_train)/len(subsample_train):.2f}x")
print(f"compress. RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

last_N_rows_train = OtherDownsamplers.get_last_N_rows(df_train, N_rows=N_last_rows)
rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(last_N_rows_train, df_test)
print(f"Last N rows ({N_last_rows=})")
print(f"data compressed {len(df_train)/len(last_N_rows_train):.2f}x")
print(f"compress. RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

# df_plotter(df, lttb_union_train, 1, time_col_name)

In [ ]:
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf

def plot_corr_matrix(df: pd.DataFrame, title: str = "Feature Correlation Matrix") -> None:
    """Plot heatmap of Pearson correlation matrix for the DataFrame features"""
    corr = df.corr()
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True, cbar_kws={"shrink": .8})
    plt.title(title)
    plt.tight_layout()
    plt.show()

def plot_autocorr(series: pd.Series, lags: int = 40, title: str = None) -> None:
    """Plot autocorrelation function (ACF) for a pandas Series"""
    plt.figure(figsize=(8, 4))
    plot_acf(series.dropna(), lags=lags, alpha=0.05)
    plt.title(title or f"Autocorrelation (up to {lags} lags)")
    plt.tight_layout()
    plt.show()

# plot_corr_matrix(df)
# plot_corr_matrix(df_downsampled)
# plot_corr_matrix(df_downsampled2)

plot_autocorr(df['act_still'], lags=10)
plot_autocorr(df_downsampled['act_still'], lags=10)
plot_autocorr(df_downsampled2['act_still'], lags=10)

In [ ]:
"""LSTM"""
# ===== Example usage =====
n_samples  = 100
n_rows     = 10
n_features = 15
X = torch.randn(n_samples, n_rows, n_features)
y = torch.randn(n_samples, 1)

# Train/val split
train_size = 80
train_X, val_X = X[:train_size], X[train_size:]
train_y, val_y = y[:train_size], y[train_size:]

# Model/optimizer/loss
model     = LSTMModel(input_size=n_features, hidden_size=64, num_layers=2, output_size=1)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Train
trainer = LSTMTrainer(model, optimizer, criterion)
trainer.fit(train_X, train_y, val_X, val_y, epochs=300)

# Inference
new_seq    = torch.randn(1, n_rows, n_features)  # batch=1, seq=10, features=15
prediction = trainer.predict(new_seq)
print("Prediction:", prediction)


In [ ]:
"""Seq2seq LSTM"""

import torch
import torch.nn as nn

class Seq2SeqLSTM(nn.Module):
    """
    Encoder-decoder LSTM for multi-step forecasting.
    Auto-regressive: generates one step at a time.
    """
    def __init__(self, input_size, hidden_size, num_layers, output_size, device='cpu'):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.device = device

        # Encoder LSTM
        self.encoder = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        # Decoder LSTM (one step at a time)
        self.decoder = nn.LSTM(output_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_input, horizon):
        """
        encoder_input: (batch, seq_len, input_size)
        horizon: number of future steps to predict
        returns: (batch, horizon, output_size)
        """
        batch_size = encoder_input.size(0)
        # Encode
        _, (h, c) = self.encoder(encoder_input)

        # Initialize decoder input as last step of encoder
        decoder_input = encoder_input[:, -1:, :]  # shape: (batch, 1, input_size)
        outputs = []

        for t in range(horizon):
            out, (h, c) = self.decoder(decoder_input, (h, c))
            step_pred = self.fc(out)  # (batch, 1, output_size)
            outputs.append(step_pred)
            decoder_input = step_pred  # feed prediction as next input

        return torch.cat(outputs, dim=1)  # (batch, horizon, output_size)

# toy data
batch_size, seq_len, n_features = 4, 10, 3
horizon = 5
X = torch.randn(batch_size, seq_len, n_features)

model = Seq2SeqLSTM(input_size=n_features, hidden_size=32, num_layers=1, output_size=n_features)
preds = model(X, horizon)  # shape: (batch, horizon, n_features)
print(preds.shape)


In [ ]:
"""Make scatterplot of compressions vs rmse"""
compression_ratio_list, rmse_list, nrmse_list = [], [], []
new_points_per_col_ratio = list(range(2, 10, 8))
new_points_per_col = [int(len(df) / ratio) for ratio in new_points_per_col_ratio]

for i, point_n in enumerate(new_points_per_col):
    compression_ratio_list, rmse_list, nrmse_list = downsample_and_evaluate(
        df, point_n, compression_ratio_list, rmse_list, nrmse_list, rdp_epsilon=0.1, method="union_rdp")

rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(df)

plt.figure(figsize=(8, 5))
# plt.scatter(compression_ratio_list, nrmse_list, label="Norm. RMSE", marker="o")
# plt.scatter(1, nrmse, label="original")

plt.scatter(compression_ratio_list, rmse_list, label="RMSE", marker="x")
plt.scatter(1, rmse, label="original")

plt.xlabel("Compression Ratio")
plt.ylabel("RMS error")
plt.title("Error vs Compression Ratio")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# """Choose good epsilon for RDP"""
# for col in df.columns:
#     if col == time_col_name:
#         continue
#     if pd.api.types.is_datetime64_any_dtype(df[time_col_name]):
#         x_vals = df[time_col_name].astype("int64")
#     else:
#         x_vals = df[time_col_name].values

#     coords = np.column_stack((x_vals, df[col].values))
#     eps1 = OtherDownsamplers.choose_rdp_epsilon(coords, method="mad", factor=1.0)
#     eps2 = OtherDownsamplers.choose_rdp_epsilon(coords, method="std", factor=1.0)
#     eps3 = OtherDownsamplers.choose_rdp_epsilon(coords, method="fraction", factor=1e-3)

#     print(f"RDP epsilon (mad): {eps1:.3f}, (std): {eps2:.3f}, (fraction): {eps3:.3f}")